# Document Restoration U-Net (Warm-Start Fine-Tuning với 28 Cặp Ảnh Thực Tế + Kiểm Chứng Blind Test `real_11.jpg` & `real_12.jpg`)

Notebook này thống nhất kiến trúc **AI-centric (`DocumentRestorationUNet`)**, tận dụng trọng số `models/restoration_best.pth` đã học và Fine-tune nối tiếp **30 epochs** trên bộ dữ liệu kết hợp:
- **Nhóm Train (`real_pairs` - 28 ảnh thực tế = 18 ảnh `real_01..10, 13..20` + 10 ảnh `IMG20261022*`)**:
  - Dùng `refine_real_clean_target` tạo `target` sạch lưu tại `datasets/processed/real_pairs/target/` và đưa vào `datasets/metadata/train.csv` với `oversample x3` (84 dòng thực tế + 183 dòng gốc = **267 mẫu train**).
  - Tuyệt đối loại trừ `real_11.jpg` và `real_12.jpg` khỏi `real_pairs` và `train.csv` (0% data leakage).
- **Nhóm Kiểm chứng Thực lực Pure U-Net (`real_11.jpg` ở `test/input/` và `real_12.jpg` ở `val/input/`)**:
  - Chế độ **Unpaired Real Evaluation (Chỉ có Input gốc, không có Target giả lập)**: không tính PSNR/SSIM với target giả; chạy **100% Pure U-Net Inference (`pred = model(input)`)** trực tiếp từ ảnh Input gốc và xuất ảnh so sánh 2 cột trung thực vào `outputs/restoration/real_blind_test/`.


In [ ]:
#@title 1. Mount Google Drive & Kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')

import torch, os, sys, zipfile, shutil, importlib, glob
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(),
      '| Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (Hãy vào Runtime > Change runtime type > T4 GPU)')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
#@title 2. Cấu hình đường dẫn, Giải nén datasets_colab_with_real.zip & Chế độ Warm-Start 30 Epochs
PROJECT_DIR = '/content/drive/MyDrive/doc-force-ai'               #@param {type:"string"}
ZIP_PATH    = '/content/drive/MyDrive/datasets_colab_with_real.zip' #@param {type:"string"}
FORCE_UNZIP = True                                                #@param {type:"boolean"}
RUN_NAME    = 'unet_3stage_real_finetune'                         #@param {type:"string"}
IMAGE_SIZE  = 512     #@param {type:"integer"}
BATCH_SIZE  = 4       #@param {type:"integer"}
EPOCHS      = 30      #@param {type:"integer"}
LR          = 5e-5    #@param {type:"number"}
VIS_EVERY   = 5       #@param {type:"integer"}
SEED        = 42

os.makedirs(PROJECT_DIR, exist_ok=True)
DATASET_ROOT = f'{PROJECT_DIR}/datasets'
MANIFEST     = f'{DATASET_ROOT}/metadata/dataset_manifest.csv'
TRAIN_CSV    = f'{DATASET_ROOT}/metadata/train.csv'
VAL_CSV      = f'{DATASET_ROOT}/metadata/val.csv'
TEST_CSV     = f'{DATASET_ROOT}/metadata/test.csv'
RUN_DIR      = f'{PROJECT_DIR}/runs/{RUN_NAME}'
MODELS_DIR   = f'{PROJECT_DIR}/models'
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

candidate_zips = [
    ZIP_PATH,
    '/content/drive/MyDrive/datasets_colab_with_real.zip',
    f'{PROJECT_DIR}/datasets_colab_with_real.zip',
    '/content/datasets_colab_with_real.zip',
    '/content/drive/MyDrive/datasets_colab_v3.zip',
    f'{PROJECT_DIR}/datasets_colab_v3.zip',
    '/content/datasets_colab_v3.zip',
]
found_zip = next((z for z in candidate_zips if os.path.exists(z)), None)

target_check_files = [
    f'{DATASET_ROOT}/processed/test/target/Anh11.jfif',
    f'{DATASET_ROOT}/processed/real_pairs/target/real_01.png',
    f'{DATASET_ROOT}/processed/real_pairs/target/real_20.png',
    f'{DATASET_ROOT}/processed/test/input/real_11.jpg',
    f'{DATASET_ROOT}/processed/val/input/real_12.jpg',
]
need_unzip = FORCE_UNZIP or (not os.path.exists(TEST_CSV)) or any(not os.path.exists(p) for p in target_check_files)

if need_unzip and found_zip:
    print(f'Đang giải nén dữ liệu mới nhất từ {found_zip} vào {PROJECT_DIR} ...')
    with zipfile.ZipFile(found_zip, 'r') as zf:
        zf.extractall(PROJECT_DIR)
    print('Giải nén hoàn tất!')
elif os.path.exists(MANIFEST):
    print(f'Đã tìm thấy dữ liệu tại: {DATASET_ROOT}')
else:
    raise FileNotFoundError(f'Không tìm thấy {MANIFEST} hoặc file zip tại {ZIP_PATH}. Hãy upload datasets_colab_with_real.zip lên Drive.')

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print('Working directory:', os.getcwd())


In [ ]:
#@title 3. Tự động cập nhật bộ mã nguồn mới nhất vào thư mục training/
os.makedirs(f'{PROJECT_DIR}/training', exist_ok=True)

RESTORATION_DATASET_CODE = 'import os\nimport csv\nimport logging\nimport random\nfrom typing import Tuple, List, Optional, Dict\nfrom PIL import Image\nimport numpy as np\nimport cv2\nimport torch\nfrom torch.utils.data import Dataset\nimport torchvision.transforms as T\n\nlogger = logging.getLogger("restoration_dataset")\n\n\ndef load_image(path: str) -> Optional[np.ndarray]:\n    """\n    Safely load an image supporting unicode paths and all common formats (.png, .jpg, .jfif, .webp).\n    Returns BGR numpy array or None on failure.\n    """\n    if not os.path.exists(path):\n        return None\n    try:\n        pil_img = Image.open(path).convert("RGB")\n        img_rgb = np.array(pil_img)\n        return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)\n    except Exception:\n        try:\n            return cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)\n        except Exception:\n            return None\n\n\ndef save_image(path: str, img: np.ndarray) -> bool:\n    """\n    Save an image safely supporting unicode paths.\n    """\n    ext = os.path.splitext(path)[1]\n    result, buf = cv2.imencode(ext if ext else ".png", img)\n    if result:\n        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)\n        with open(path, "wb") as f:\n            f.write(buf)\n        return True\n    return False\n\n\ndef unsharp_mask(img: np.ndarray, amount: float = 0.8, radius: float = 1.0) -> np.ndarray:\n    """\n    Apply Unsharp Masking filter to sharpen text edges and contrast.\n    """\n    blurred = cv2.GaussianBlur(img, (0, 0), radius)\n    sharpened = cv2.addWeighted(img, 1.0 + amount, blurred, -amount, 0)\n    return np.clip(sharpened, 0, 255).astype(np.uint8)\n\n\ndef resize_with_padding(\n    img: np.ndarray,\n    target_size: Tuple[int, int] = (512, 512)\n) -> Tuple[np.ndarray, Tuple[int, int, int, int]]:\n    """\n    Resize image keeping aspect ratio and pad with zeros to reach target_size.\n\n    Returns:\n        canvas: padded image of shape (target_h, target_w, 3)\n        pad_info: (pad_top, pad_left, new_h, new_w) for later unpadding\n    """\n    h, w = img.shape[:2]\n    target_h, target_w = target_size\n    scale = min(target_w / w, target_h / h)\n    new_w, new_h = int(w * scale), int(h * scale)\n\n    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)\n\n    canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)\n    pad_top = (target_h - new_h) // 2\n    pad_left = (target_w - new_w) // 2\n    canvas[pad_top:pad_top + new_h, pad_left:pad_left + new_w] = resized\n\n    return canvas, (pad_top, pad_left, new_h, new_w)\n\n\ndef remove_padding(\n    img: np.ndarray,\n    pad_info: Tuple[int, int, int, int],\n    original_size: Tuple[int, int]\n) -> np.ndarray:\n    """\n    Remove padding and resize back to original dimensions.\n\n    Args:\n        img: padded image of shape (target_h, target_w, 3)\n        pad_info: (pad_top, pad_left, new_h, new_w) from resize_with_padding\n        original_size: (orig_h, orig_w) to resize back to\n    """\n    pad_top, pad_left, new_h, new_w = pad_info\n    cropped = img[pad_top:pad_top + new_h, pad_left:pad_left + new_w]\n    orig_h, orig_w = original_size\n    restored = cv2.resize(cropped, (orig_w, orig_h), interpolation=cv2.INTER_LANCZOS4)\n    return restored\n\n\ndef pad_if_needed(img: np.ndarray, patch_size: Tuple[int, int]) -> np.ndarray:\n    """Pad image using BORDER_REFLECT if any dimension is smaller than patch_size."""\n    h, w = img.shape[:2]\n    patch_h, patch_w = patch_size\n    pad_h = max(0, patch_h - h)\n    pad_w = max(0, patch_w - w)\n    if pad_h > 0 or pad_w > 0:\n        pad_top = pad_h // 2\n        pad_bottom = pad_h - pad_top\n        pad_left = pad_w // 2\n        pad_right = pad_w - pad_left\n        return cv2.copyMakeBorder(img, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_REFLECT)\n    return img\n\n\ndef crop_patch(\n    deg_img: np.ndarray,\n    clean_img: np.ndarray,\n    patch_size: Tuple[int, int] = (512, 512),\n    is_train: bool = True\n) -> Tuple[np.ndarray, np.ndarray]:\n    """\n    Extract matching patch from degraded and clean images.\n    Uses RandomCrop when is_train=True, CenterCrop when is_train=False.\n    Pads with BORDER_REFLECT if image is smaller than patch_size in either dimension.\n    """\n    deg_pad = pad_if_needed(deg_img, patch_size)\n    clean_pad = pad_if_needed(clean_img, patch_size)\n\n    h, w = deg_pad.shape[:2]\n    patch_h, patch_w = patch_size\n\n    if is_train:\n        max_top = max(0, h - patch_h)\n        max_left = max(0, w - patch_w)\n        top = random.randint(0, max_top) if max_top > 0 else 0\n        left = random.randint(0, max_left) if max_left > 0 else 0\n    else:\n        top = max(0, (h - patch_h) // 2)\n        left = max(0, (w - patch_w) // 2)\n\n    deg_crop = deg_pad[top:top + patch_h, left:left + patch_w]\n    clean_crop = clean_pad[top:top + patch_h, left:left + patch_w]\n    return deg_crop, clean_crop\n\n\nGEOMETRIC_DISTORTION_KEYWORDS = (\n    "bookbinding",\n    "perspective",\n    "opticaldistortion",\n    "centercrop",\n    "bindingsandfasteners",\n)\n\n\ndef _sauvola_threshold_integral(\n    img_f: np.ndarray,\n    window_size: int = 15,\n    k: float = 0.15,\n    r: float = 128.0,\n) -> np.ndarray:\n    """\n    Compute local Sauvola threshold T(x,y) = mu(x,y) * [1 + k * (sigma(x,y) / R - 1)]\n    in O(1) time per pixel using OpenCV integral images.\n    """\n    pad = window_size // 2\n    padded = cv2.copyMakeBorder(img_f, pad, pad, pad, pad, cv2.BORDER_REFLECT).astype(np.float64)\n    s, s2 = cv2.integral2(padded)\n    w = window_size\n    area = float(w * w)\n\n    win_sum = s[w:, w:] - s[:-w, w:] - s[w:, :-w] + s[:-w, :-w]\n    win_sqsum = s2[w:, w:] - s2[:-w, w:] - s2[w:, :-w] + s2[:-w, :-w]\n\n    mu = win_sum / area\n    var = np.maximum(win_sqsum / area - (mu * mu), 0.0)\n    sigma = np.sqrt(var)\n\n    thresh = mu * (1.0 + k * ((sigma / r) - 1.0))\n    return thresh.astype(np.float32)\n\n\ndef clean_target_normalization(\n    clean_gray: np.ndarray,\n    use_nlm: bool = False,\n    gamma: float = 0.75,\n) -> np.ndarray:\n    """\n    Master Ground-Truth Generator: 5-step Soft Anti-Aliased Document Mastering\n    ("Multi-Scale Morphological + Sauvola Soft Matting").\n\n    1. Edge-Preserving Pre-Denoising (Bilateral Filter or Fast Non-Local Means).\n    2. Multi-Scale Morphological Background Estimation (21x21 & 45x45 elliptical closing + Gaussian sigma=15).\n    3. Multi-Scale Sauvola Soft Alpha Matting (O(1) integral image with w=15,k=0.15 & w=31,k=0.20 + Soft Ramp).\n    4. Topology-Aware Micro-Speckle Removal (removes isolated dust <= 2px with min_intensity > 140).\n    5. Foreground Unsharp Boost (sharpens only stroke pixels < 245).\n    """\n    if clean_gray is None or clean_gray.size == 0:\n        return clean_gray\n    if clean_gray.ndim == 3:\n        clean_gray = cv2.cvtColor(clean_gray, cv2.COLOR_BGR2GRAY)\n\n    # Step 1: Edge-Preserving Pre-Denoising\n    if use_nlm:\n        denoised = cv2.fastNlMeansDenoising(\n            clean_gray, None, h=6, templateWindowSize=7, searchWindowSize=21\n        )\n    else:\n        denoised = cv2.bilateralFilter(clean_gray, d=5, sigmaColor=25, sigmaSpace=25)\n\n    # Step 2: Multi-Scale Morphological Background Estimation\n    kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))\n    kernel_large = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (45, 45))\n    bg_small = cv2.morphologyEx(denoised, cv2.MORPH_CLOSE, kernel_small)\n    bg_large = cv2.morphologyEx(denoised, cv2.MORPH_CLOSE, kernel_large)\n    bg = np.maximum(bg_small, bg_large)\n    bg = cv2.GaussianBlur(bg, (0, 0), sigmaX=15, sigmaY=15)\n\n    norm = np.clip(\n        (denoised.astype(np.float32) / np.maximum(bg.astype(np.float32), 15.0)) * 255.0,\n        0.0,\n        255.0,\n    )\n\n    # Step 3: Multi-Scale Sauvola Soft Alpha Matting\n    t_small = _sauvola_threshold_integral(norm, window_size=15, k=0.15, r=128.0)\n    t_med = _sauvola_threshold_integral(norm, window_size=31, k=0.20, r=128.0)\n    t_sauvola = np.maximum(t_small, t_med)\n\n    diff = norm - t_sauvola\n    bg_mask = (diff > 6.0) | (norm > 225.0)\n    fg_mask = ~bg_mask\n\n    matted_f = np.full_like(norm, 255.0, dtype=np.float32)\n    if np.any(fg_mask):\n        fg_vals = norm[fg_mask]\n        black_floor = float(np.clip(np.percentile(fg_vals, 18), 20.0, 145.0))\n        upper_bound = np.minimum(t_sauvola + 6.0, 225.0)\n        span = np.maximum(upper_bound - black_floor, 20.0)\n\n        # Compute foreground opacity alpha in [0, 1], boost with gamma < 1.0, and map to [0, 255]\n        alpha = np.clip((upper_bound - norm) / span, 0.0, 1.0)\n        alpha_boosted = np.power(alpha, gamma)\n        matted_f = np.where(bg_mask, 255.0, 255.0 * (1.0 - alpha_boosted))\n\n    matted = np.clip(matted_f, 0.0, 255.0).astype(np.uint8)\n\n    # Step 4: Topology-Aware Micro-Speckle Removal\n    dark_mask = (matted < 200).astype(np.uint8)\n    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(dark_mask, connectivity=8)\n    for lbl in range(1, num_labels):\n        area = stats[lbl, cv2.CC_STAT_AREA]\n        if area <= 2:\n            x0 = stats[lbl, cv2.CC_STAT_LEFT]\n            y0 = stats[lbl, cv2.CC_STAT_TOP]\n            w0 = stats[lbl, cv2.CC_STAT_WIDTH]\n            h0 = stats[lbl, cv2.CC_STAT_HEIGHT]\n            roi_labels = labels[y0:y0 + h0, x0:x0 + w0]\n            roi_img = matted[y0:y0 + h0, x0:x0 + w0]\n            comp_mask = roi_labels == lbl\n            if int(np.min(roi_img[comp_mask])) > 140:\n                roi_img[comp_mask] = 255\n\n    # Step 5: Foreground Unsharp Boost (only on stroke pixels < 245)\n    sharpened = unsharp_mask(matted, amount=0.65, radius=1.0)\n    out = np.where(matted < 245, sharpened, 255).astype(np.uint8)\n    return out\n\n\n# Alias for backward compatibility across training and evaluation\nnormalize_clean_target = clean_target_normalization\n\n\ndef apply_hard_augmentation(\n    deg_patch: np.ndarray,\n    clean_patch: np.ndarray,\n    is_real_pair: bool = False,\n) -> Tuple[np.ndarray, np.ndarray]:\n    """\n    On-the-fly hard augmentation for training patches:\n    - 50% probability: synchronous HorizontalFlip on both degraded and clean patches.\n    - For synthetic/standard samples (`is_real_pair=False`), 30% probability: synthetic local shadow\n      patch or mild Gaussian blur on degraded patch.\n    - For `real_pairs` (`is_real_pair=True`), preserves authentic camera degradation (only horizontal flip).\n    """\n    # 1. 50% chance: synchronous HorizontalFlip\n    if random.random() < 0.5:\n        deg_patch = cv2.flip(deg_patch, 1)\n        clean_patch = cv2.flip(clean_patch, 1)\n\n    if is_real_pair:\n        return deg_patch, clean_patch\n\n    # 2. 30% chance: mild Gaussian Blur and/or Synthetic Local Shadow on deg_patch\n    if random.random() < 0.3:\n        aug_mode = random.choice(["blur", "shadow", "both"])\n        deg_f = deg_patch.astype(np.float32)\n\n        if aug_mode in ("blur", "both"):\n            ksize = random.choice([3, 5])\n            sigma = random.uniform(0.6, 1.3)\n            deg_f = cv2.GaussianBlur(deg_f, (ksize, ksize), sigmaX=sigma, sigmaY=sigma)\n\n        if aug_mode in ("shadow", "both"):\n            h, w = deg_f.shape[:2]\n            # Create a smooth local shadow mask darkening a random region by factor [0.45, 0.75]\n            shadow_mask = np.ones((h, w), dtype=np.float32)\n            cx = random.randint(int(w * 0.2), int(w * 0.8))\n            cy = random.randint(int(h * 0.2), int(h * 0.8))\n            rx = random.randint(int(w * 0.25), int(w * 0.6))\n            ry = random.randint(int(h * 0.25), int(h * 0.6))\n            angle = random.uniform(0, 180)\n            darkness = random.uniform(0.45, 0.75)\n            cv2.ellipse(shadow_mask, (cx, cy), (rx, ry), angle, 0, 360, darkness, -1)\n            shadow_mask = cv2.GaussianBlur(shadow_mask, (0, 0), sigmaX=25, sigmaY=25)\n            deg_f = deg_f * shadow_mask\n\n        deg_patch = np.clip(deg_f, 0, 255).astype(np.uint8)\n\n    return deg_patch, clean_patch\n\n\ndef infer_single_image_tiled(\n    model: torch.nn.Module,\n    img_gray: np.ndarray,\n    device: torch.device,\n    tile_size: int = 512,\n    overlap: int = 128,\n    refine_glyphs: bool = False,\n) -> np.ndarray:\n    """\n    Sliding Window Tiling Inference with 2D Gaussian feather blending.\n    Runs 100% pure neural network forward passes through DocumentRestorationUNet.\n    refine_glyphs: parameter kept for compatibility; no external morphological/heuristic\n    filter or background clipping is performed (100% pure neural network output).\n    """\n    h_orig, w_orig = img_gray.shape[:2]\n\n    # On CPU, if image is ultra-high resolution (>1600px, e.g. 9MP-12MP camera raw),\n    # run tiled U-Net at 1280px working scale (YOLOv8 native high-res) and upsample back\n    if device.type == "cpu" and max(h_orig, w_orig) > 1600 and tile_size <= 512:\n        scale = 1280.0 / float(max(h_orig, w_orig))\n        w_work = max(64, int(round(w_orig * scale)))\n        h_work = max(64, int(round(h_orig * scale)))\n        img_work = cv2.resize(img_gray, (w_work, h_work), interpolation=cv2.INTER_AREA)\n        pred_work = infer_single_image_tiled(\n            model, img_work, device, tile_size=tile_size, overlap=min(overlap, 64), refine_glyphs=refine_glyphs\n        )\n        return cv2.resize(pred_work, (w_orig, h_orig), interpolation=cv2.INTER_CUBIC)\n\n    # Small image fast path: pad to multiple of 8 and run single forward pass\n    if h_orig <= tile_size and w_orig <= tile_size:\n        pad_h = (8 - (h_orig % 8)) % 8\n        pad_w = (8 - (w_orig % 8)) % 8\n        if pad_h > 0 or pad_w > 0:\n            img_pad = cv2.copyMakeBorder(img_gray, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)\n        else:\n            img_pad = img_gray\n        tensor_in = torch.from_numpy(img_pad).unsqueeze(0).unsqueeze(0).float().to(device) / 255.0\n        with torch.no_grad():\n            pred = model(tensor_in)\n        out = (torch.clamp(pred, 0.0, 1.0).squeeze().cpu().numpy() * 255.0).astype(np.uint8)\n        return out[:h_orig, :w_orig]\n\n    # Pad any dimension smaller than tile_size\n    pad_h = max(0, tile_size - h_orig)\n    pad_w = max(0, tile_size - w_orig)\n    if pad_h > 0 or pad_w > 0:\n        img_pad = cv2.copyMakeBorder(img_gray, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)\n    else:\n        img_pad = img_gray\n    h_pad, w_pad = img_pad.shape[:2]\n\n    # 2D Gaussian feather blending window (size: tile_size x tile_size)\n    coords = np.linspace(-1.0, 1.0, tile_size, dtype=np.float32)\n    g1d = np.exp(-2.0 * (coords ** 2))\n    weight_2d = np.outer(g1d, g1d).astype(np.float32)\n\n    stride = max(1, tile_size - overlap)\n    y_starts = list(range(0, h_pad - tile_size + 1, stride))\n    if not y_starts or y_starts[-1] != h_pad - tile_size:\n        y_starts.append(h_pad - tile_size)\n\n    x_starts = list(range(0, w_pad - tile_size + 1, stride))\n    if not x_starts or x_starts[-1] != w_pad - tile_size:\n        x_starts.append(w_pad - tile_size)\n\n    acc_pred = np.zeros((h_pad, w_pad), dtype=np.float32)\n    acc_weight = np.zeros((h_pad, w_pad), dtype=np.float32)\n\n    with torch.no_grad():\n        for y in y_starts:\n            for x in x_starts:\n                tile = img_pad[y:y + tile_size, x:x + tile_size]\n                tile_t = torch.from_numpy(tile).unsqueeze(0).unsqueeze(0).float().to(device) / 255.0\n                pred_t = model(tile_t)\n                pred_np = torch.clamp(pred_t, 0.0, 1.0).squeeze().cpu().numpy()\n                acc_pred[y:y + tile_size, x:x + tile_size] += pred_np * weight_2d\n                acc_weight[y:y + tile_size, x:x + tile_size] += weight_2d\n\n    merged = acc_pred / np.maximum(acc_weight, 1e-6)\n    return (np.clip(merged[:h_orig, :w_orig], 0.0, 1.0) * 255.0).astype(np.uint8)\n\n\nclass DocumentRestorationDataset(Dataset):\n    """\n    Dataset loader for Document Image Restoration model.\n    Reads paired degraded/clean images from a CSV metadata file.\n    Supports Grayscale 1-channel patch-based training (RandomCrop) and validation (CenterCrop).\n\n    Two CSV formats are supported (auto-detected from the header):\n\n    1. Manifest (``datasets/metadata/dataset_manifest.csv``) - many inputs -> one target.\n       Columns: split, original_id, degradation_type, variant_id, input_path, target_path.\n       Paths are resolved relative to ``root_dir`` (default: parent of the CSV\'s folder, i.e.\n       the ``datasets`` directory). Input and target names are NOT assumed to match.\n       Use ``split`` (\'train\'/\'val\'/\'test\') and ``degradation_types`` to filter rows.\n\n    2. Legacy: columns input, target (+ optional source_id, degradation, blur_*...).\n    """\n\n    def __init__(\n        self,\n        metadata_csv: str,\n        image_size: Tuple[int, int] = (512, 512),\n        patch_size: Optional[Tuple[int, int]] = None,\n        transform: Optional[T.Compose] = None,\n        return_metadata: bool = False,\n        split: Optional[str] = None,\n        root_dir: Optional[str] = None,\n        degradation_types: Optional[List[str]] = None,\n        target_cache_size: int = 64,\n        is_train: Optional[bool] = None,\n        filter_geometric: Optional[bool] = None,\n        include_unpaired: bool = False,\n    ):\n        if patch_size is not None:\n            self.patch_size = patch_size\n        else:\n            self.patch_size = image_size if isinstance(image_size, tuple) else (image_size, image_size)\n        self.image_size = self.patch_size\n        self.include_unpaired = include_unpaired\n\n        # Infer effective split (\'train\', \'val\', \'test\') when split is not explicitly passed\n        csv_basename = os.path.basename(metadata_csv).lower() if metadata_csv else ""\n        if split is not None:\n            effective_split = split.strip().lower()\n        elif "test" in csv_basename:\n            effective_split = "test"\n        elif "val" in csv_basename:\n            effective_split = "val"\n        elif "train" in csv_basename:\n            effective_split = "train"\n        else:\n            effective_split = "train" if (is_train is None or is_train) else "val"\n        self.effective_split = effective_split\n\n        if is_train is not None:\n            self.is_train = is_train\n        else:\n            self.is_train = (effective_split == "train")\n\n        if filter_geometric is not None:\n            self.filter_geometric = filter_geometric\n        else:\n            self.filter_geometric = effective_split in ("train", "val")\n\n        self.return_metadata = return_metadata\n        self.target_cache_size = target_cache_size\n        self._target_cache: Dict[str, np.ndarray] = {}\n\n        # Read CSV metadata\n        self.pairs: List[Dict[str, str]] = []\n        self.unpaired_samples: List[Dict[str, str]] = []\n        self.total_rows = 0\n        self.missing_count = 0\n        self.filtered_count = 0\n        self.unpaired_count = 0\n\n        if transform is not None:\n            self.transform = transform\n        else:\n            self.transform = T.Compose([\n                T.ToTensor(),  # converts [H,W] uint8 to [1,H,W] float [0,1]\n            ])\n\n        if not os.path.exists(metadata_csv):\n            logger.error(f"Metadata CSV file not found: \'{metadata_csv}\'")\n            return\n\n        if root_dir is None:\n            root_dir = os.path.dirname(os.path.dirname(os.path.abspath(metadata_csv)))\n        wanted_types = set(degradation_types) if degradation_types else None\n\n        with open(metadata_csv, "r", encoding="utf-8-sig", newline="") as f:\n            reader = csv.DictReader(f)\n            is_manifest = "input_path" in (reader.fieldnames or [])\n            for row in reader:\n                if is_manifest:\n                    row_split = row.get("split", "").strip()\n                    if split is not None and row_split != split:\n                        continue\n                    deg_str = row.get("degradation_type", "").strip()\n                    if wanted_types is not None and deg_str not in wanted_types:\n                        continue\n                    self.total_rows += 1\n                    raw_in = (row.get("input_path") or "").strip()\n                    raw_tgt = (row.get("target_path") or "").strip()\n                    input_path = os.path.normpath(os.path.join(root_dir, raw_in)) if raw_in else ""\n                    target_path = os.path.normpath(os.path.join(root_dir, raw_tgt)) if raw_tgt else ""\n                    src_id = row.get("original_id", "").strip()\n                    active_split = (row_split or effective_split).lower()\n                else:\n                    self.total_rows += 1\n                    raw_in = (row.get("input") or "").strip()\n                    raw_tgt = (row.get("target") or "").strip()\n                    input_path = os.path.normpath(raw_in) if raw_in else ""\n                    target_path = os.path.normpath(raw_tgt) if raw_tgt else ""\n                    deg_str = (row.get("degradation") or row.get("blur_level") or "").strip()\n                    src_id = (row.get("source_id") or row.get("original_id") or "").strip()\n                    active_split = (row.get("split") or effective_split).strip().lower()\n\n                in_stem = os.path.splitext(os.path.basename(input_path))[0].lower() if input_path else ""\n                is_blind_real = (\n                    in_stem in ("real_11", "real_12")\n                    or deg_str.lower() in ("unpaired_real", "blind_real")\n                    or (raw_tgt == "" and bool(raw_in))\n                )\n\n                # Never allow blind evaluation images (real_11, real_12) into the train split\n                if active_split == "train" and (in_stem in ("real_11", "real_12") or is_blind_real):\n                    self.filtered_count += 1\n                    continue\n\n                if is_blind_real:\n                    if input_path and os.path.exists(input_path):\n                        self.unpaired_count += 1\n                        unpaired_rec = {\n                            "input": input_path,\n                            "target": "",\n                            "source_id": src_id or in_stem,\n                            "degradation": deg_str or "unpaired_real",\n                            "split": active_split,\n                            "is_real_pair": False,\n                            "is_unpaired": True,\n                        }\n                        self.unpaired_samples.append(unpaired_rec)\n                        if self.include_unpaired and not self.is_train:\n                            self.pairs.append(unpaired_rec)\n                    else:\n                        self.missing_count += 1\n                    continue\n\n                if not (input_path and target_path and os.path.exists(input_path) and os.path.exists(target_path)):\n                    self.missing_count += 1\n                    continue\n\n                # Filter geometric distortions (BookBinding, Perspective, OpticalDistortion) in train & val\n                # and filter Anh06 in train (open book on wooden table)\n                if self.filter_geometric and active_split in ("train", "val"):\n                    check_text = f"{os.path.basename(input_path)} {deg_str}".lower()\n                    if any(kw in check_text for kw in GEOMETRIC_DISTORTION_KEYWORDS):\n                        self.filtered_count += 1\n                        continue\n                    if active_split == "train":\n                        id_text = f"{src_id} {os.path.basename(input_path)} {os.path.basename(target_path)}".lower()\n                        if "anh06" in id_text:\n                            self.filtered_count += 1\n                            continue\n\n                is_real_pair = (\n                    "real_pairs" in input_path.replace("\\\\", "/").lower()\n                    or "real_pairs" in target_path.replace("\\\\", "/").lower()\n                    or deg_str.lower() in ("real_pairs", "realcamera")\n                )\n\n                if is_manifest:\n                    self.pairs.append({\n                        "input": input_path,\n                        "target": target_path,\n                        "source_id": src_id,\n                        "original_id": src_id,\n                        "degradation": deg_str,\n                        "variant_id": row.get("variant_id", ""),\n                        "split": row.get("split", ""),\n                        "is_real_pair": is_real_pair,\n                        "is_unpaired": False,\n                    })\n                else:\n                    self.pairs.append({\n                        "input": input_path,\n                        "target": target_path,\n                        "source_id": src_id,\n                        "degradation": deg_str,\n                        "is_real_pair": is_real_pair,\n                        "is_unpaired": False,\n                    })\n\n        if self.missing_count > 0:\n            logger.warning(\n                f"Missing {self.missing_count}/{self.total_rows} image pair(s) in \'{metadata_csv}\'. "\n                f"Loaded {len(self.pairs)} valid pair(s)."\n            )\n        elif self.total_rows == 0:\n            logger.warning(f"Metadata CSV \'{metadata_csv}\' is empty (split={split}).")\n        elif self.filtered_count > 0 or self.unpaired_count > 0:\n            logger.info(\n                f"Split=\'{effective_split}\': Loaded {len(self.pairs)} paired sample(s), "\n                f"{self.unpaired_count} unpaired blind-eval sample(s), "\n                f"filtered {self.filtered_count} geometric/non-flat pair(s)."\n            )\n\n    def __len__(self) -> int:\n        return len(self.pairs)\n\n    def _load_target(self, path: str, skip_normalization: bool = False) -> np.ndarray:\n        """\n        Load full-resolution clean target image as Grayscale (cached).\n        For `real_pairs` targets (already carefully refined with bleed-through & watermark removal),\n        skips re-running simple `clean_target_normalization`.\n        """\n        cached = self._target_cache.get(path)\n        if cached is not None:\n            return cached\n        clean_bgr = load_image(path)\n        if clean_bgr is None:\n            raise FileNotFoundError(f"Failed to load clean target image: {path}")\n        clean_gray = cv2.cvtColor(clean_bgr, cv2.COLOR_BGR2GRAY)\n        is_real_target = skip_normalization or ("real_pairs" in path.replace("\\\\", "/").lower())\n        if not is_real_target:\n            clean_gray = clean_target_normalization(clean_gray)\n        if len(self._target_cache) < self.target_cache_size:\n            self._target_cache[path] = clean_gray\n        return clean_gray\n\n    def __getitem__(self, idx: int):\n        pair = self.pairs[idx]\n        is_real_pair = bool(pair.get("is_real_pair", False))\n        is_unpaired = bool(pair.get("is_unpaired", False)) or not pair.get("target")\n\n        # Load degraded image as numpy BGR -> Grayscale\n        deg_bgr = load_image(pair["input"])\n        if deg_bgr is None:\n            raise FileNotFoundError(f"Failed to load degraded input image: {pair[\'input\']}")\n\n        deg_gray = cv2.cvtColor(deg_bgr, cv2.COLOR_BGR2GRAY)\n\n        if is_unpaired:\n            deg_patch, _ = crop_patch(deg_gray, deg_gray, patch_size=self.patch_size, is_train=False)\n            deg_tensor = self.transform(deg_patch)\n            if deg_tensor.ndim == 2:\n                deg_tensor = deg_tensor.unsqueeze(0)\n            if self.return_metadata:\n                return deg_tensor, None, pair\n            return deg_tensor, deg_tensor\n\n        clean_gray = self._load_target(pair["target"], skip_normalization=is_real_pair)\n\n        # Align dimensions if input and clean target differ in resolution (e.g. LongestMaxSize)\n        # Always resize degraded input to match clean target resolution\n        if deg_gray.shape[:2] != clean_gray.shape[:2]:\n            deg_gray = cv2.resize(deg_gray, (clean_gray.shape[1], clean_gray.shape[0]), interpolation=cv2.INTER_CUBIC)\n\n        # Patch-based extraction at full resolution (RandomCrop for train, CenterCrop for eval)\n        deg_patch, clean_patch = crop_patch(deg_gray, clean_gray, patch_size=self.patch_size, is_train=self.is_train)\n\n        # On-the-fly Hard Augmentation (HorizontalFlip 50%; skip synthetic blur/shadow for real_pairs)\n        if self.is_train:\n            deg_patch, clean_patch = apply_hard_augmentation(\n                deg_patch, clean_patch, is_real_pair=is_real_pair\n            )\n\n        # Apply transforms: converts uint8 [0,255] [H,W] to float [0,1] [1,H,W]\n        deg_tensor = self.transform(deg_patch)\n        clean_tensor = self.transform(clean_patch)\n\n        # Ensure tensor is shape [1, H, W]\n        if deg_tensor.ndim == 2:\n            deg_tensor = deg_tensor.unsqueeze(0)\n        if clean_tensor.ndim == 2:\n            clean_tensor = clean_tensor.unsqueeze(0)\n\n        if self.return_metadata:\n            return deg_tensor, clean_tensor, pair\n\n        return deg_tensor, clean_tensor\n\n\n'

TRAIN_RESTORATION_CODE = 'import os\nimport sys\nimport argparse\nimport math\nimport json\nimport logging\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport torch.optim as optim\nfrom torch.utils.data import DataLoader\n\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))\nfrom training.restoration_dataset import DocumentRestorationDataset\n\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")\nlogger = logging.getLogger("train_restoration")\n\n\nclass ResidualBlock(nn.Module):\n    """Residual block with two 3x3 convolutions and skip connection."""\n\n    def __init__(self, channels: int):\n        super().__init__()\n        self.conv = nn.Sequential(\n            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(channels),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),\n            nn.BatchNorm2d(channels),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        return x + self.conv(x)\n\n\nclass SobelLoss(nn.Module):\n    """Multi-Scale Sobel Gradient Loss to enforce sharp text edge reconstruction across scales."""\n\n    def __init__(self):\n        super().__init__()\n        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]], dtype=torch.float32).view(1, 1, 3, 3)\n        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]], dtype=torch.float32).view(1, 1, 3, 3)\n        self.register_buffer("sobel_x", sobel_x)\n        self.register_buffer("sobel_y", sobel_y)\n\n    def _grad_l1(self, p: torch.Tensor, t: torch.Tensor) -> torch.Tensor:\n        p_gx = F.conv2d(p, self.sobel_x, padding=1)\n        p_gy = F.conv2d(p, self.sobel_y, padding=1)\n        t_gx = F.conv2d(t, self.sobel_x, padding=1)\n        t_gy = F.conv2d(t, self.sobel_y, padding=1)\n        return F.l1_loss(p_gx, t_gx) + F.l1_loss(p_gy, t_gy)\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        if pred.shape[1] == 3:\n            pred_g = 0.299 * pred[:, 0:1] + 0.587 * pred[:, 1:2] + 0.114 * pred[:, 2:3]\n            target_g = 0.299 * target[:, 0:1] + 0.587 * target[:, 1:2] + 0.114 * target[:, 2:3]\n        else:\n            pred_g = pred\n            target_g = target\n\n        loss_scale1 = self._grad_l1(pred_g, target_g)\n        if pred_g.shape[-2] >= 8 and pred_g.shape[-1] >= 8:\n            pred_half = F.avg_pool2d(pred_g, kernel_size=2, stride=2)\n            target_half = F.avg_pool2d(target_g, kernel_size=2, stride=2)\n            loss_scale2 = self._grad_l1(pred_half, target_half)\n            return loss_scale1 + 0.5 * loss_scale2\n        return loss_scale1\n\n\nclass BimodalStrokeLoss(nn.Module):\n    """\n    Penalizes washed-out gray predictions on text strokes while preserving soft anti-aliased\n    target edges. Forces stroke cores to be crisp dark ink rather than muddy mid-gray.\n    """\n\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred = pred.float()\n        target = target.float()\n\n        # 1. Bimodal distribution alignment: match p * (1 - p) of the clean target\n        pred_bimodal = pred * (1.0 - pred)\n        target_bimodal = target * (1.0 - target)\n        loss_bimodal = F.l1_loss(pred_bimodal, target_bimodal)\n\n        # 2. Asymmetric stroke darkening penalty: penalize faded/gray predictions (pred > target) on stroke pixels\n        stroke_mask = (target < 0.65).float()\n        faded_stroke_penalty = torch.mean(stroke_mask * (F.relu(pred - target) ** 2))\n\n        return loss_bimodal + 2.0 * faded_stroke_penalty\n\n\nclass SSIMLoss(nn.Module):\n    """Differentiable SSIM Loss (1 - SSIM) using an 11x11 Gaussian window."""\n\n    def __init__(self, window_size: int = 11, sigma: float = 1.5):\n        super().__init__()\n        self.window_size = window_size\n        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2\n        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))\n        g = g / g.sum()\n        window_2d = (g.unsqueeze(1) @ g.unsqueeze(0)).unsqueeze(0).unsqueeze(0)\n        self.register_buffer("window", window_2d)\n        self.c1 = 0.01 ** 2\n        self.c2 = 0.03 ** 2\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred = pred.float()\n        target = target.float()\n        channels = pred.shape[1]\n        win = self.window.expand(channels, 1, self.window_size, self.window_size)\n        pad = self.window_size // 2\n\n        mu1 = F.conv2d(pred, win, padding=pad, groups=channels)\n        mu2 = F.conv2d(target, win, padding=pad, groups=channels)\n        mu1_sq = mu1 * mu1\n        mu2_sq = mu2 * mu2\n        mu1_mu2 = mu1 * mu2\n\n        sigma1_sq = F.conv2d(pred * pred, win, padding=pad, groups=channels) - mu1_sq\n        sigma2_sq = F.conv2d(target * target, win, padding=pad, groups=channels) - mu2_sq\n        sigma12 = F.conv2d(pred * target, win, padding=pad, groups=channels) - mu1_mu2\n\n        ssim_map = ((2.0 * mu1_mu2 + self.c1) * (2.0 * sigma12 + self.c2)) / (\n            (mu1_sq + mu2_sq + self.c1) * (sigma1_sq + sigma2_sq + self.c2) + 1e-8\n        )\n        return torch.clamp(1.0 - ssim_map.mean(), min=0.0)\n\n\nclass BleedThroughSuppressionLoss(nn.Module):\n    """\n    Penalizes residual gray predictions (pred < 0.92) in regions where target > 0.95\n    (pure white background or back-side bleed-through/watermark regions that must be erased).\n    Teaches U-Net to distinguish and suppress back-side bleed-through without affecting front strokes.\n    """\n\n    def __init__(self, target_thresh: float = 0.95, pred_floor: float = 0.92):\n        super().__init__()\n        self.target_thresh = target_thresh\n        self.pred_floor = pred_floor\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred = pred.float()\n        target = target.float()\n        bg_or_bleed_mask = (target > self.target_thresh).float()\n        # Heavy quadratic penalty when U-Net predicts gray < 0.92 on clean background/bleed-through regions\n        below_floor = F.relu(self.pred_floor - pred)\n        white_gap = F.relu(target - pred)\n        return torch.mean(bg_or_bleed_mask * (10.0 * (below_floor ** 2) + 0.5 * white_gap))\n\n\nclass DocumentCleanLoss(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.sobel = SobelLoss()\n        self.ssim = SSIMLoss()\n        self.bimodal = BimodalStrokeLoss()\n        self.bleed_suppression = BleedThroughSuppressionLoss(target_thresh=0.95, pred_floor=0.92)\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        pred = pred.float()\n        target = target.float()\n\n        # Trọng số liên tục ưu tiên bảo vệ nét chữ (không ép 1.5x cho nền trắng để tránh over-bleaching)\n        stroke_weight = 1.0 + 6.0 * ((1.0 - target) ** 2)\n\n        # 1. Weighted Charbonnier Loss\n        charbonnier = torch.sqrt((pred - target) ** 2 + 1e-6)\n        loss_charbonnier = torch.mean(stroke_weight * charbonnier)\n\n        # 2. Weighted BCE Loss\n        pred_clamped = torch.clamp(pred, 1e-6, 1.0 - 1e-6)\n        bce_map = F.binary_cross_entropy(pred_clamped, target, reduction="none")\n        loss_bce = torch.mean(stroke_weight * bce_map)\n\n        # 3. Multi-Scale Sobel Edge Gradient Loss (0.8) để tái tạo sắc nét biên chữ khi mờ hoặc trong bóng râm\n        loss_sobel = self.sobel(pred, target)\n\n        # 4. SSIM Loss (0.2) bảo toàn cấu trúc cục bộ của ký tự\n        loss_ssim = self.ssim(pred, target)\n\n        # 5. BimodalStrokeLoss (0.25) phạt vùng chữ bị xám mờ, tăng tương phản nét chữ cho YOLOv8\n        loss_bimodal = self.bimodal(pred, target)\n\n        # 6. BleedThroughSuppressionLoss (0.35) phạt nặng nếu vùng target > 0.95 mà U-Net xuất < 0.92\n        loss_bleed = self.bleed_suppression(pred, target)\n\n        total_loss = (\n            loss_charbonnier\n            + 0.3 * loss_bce\n            + 0.8 * loss_sobel\n            + 0.2 * loss_ssim\n            + 0.25 * loss_bimodal\n            + 0.35 * loss_bleed\n        )\n        return total_loss\n\n\nclass DocumentRestorationUNet(nn.Module):\n    """\n    3-Stage U-Net architecture for Multi-Degradation Document Image Restoration.\n    Accepts [B, 1, H, W] input, computes internal Local Illumination Normalization `x_norm`,\n    concatenates `[x, x_norm]` into 2 channels for `enc1`, and uses Sigmoid output activation\n    with +0.2 bias init to preserve faint/blurred text strokes.\n    """\n\n    def __init__(self, in_channels: int = 1, out_channels: int = 1):\n        super().__init__()\n        self.in_channels = in_channels\n        enc_in_channels = in_channels * 2\n\n        # Encoder level 1 (receives concatenated [x, x_norm] -> 2 channels when in_channels=1)\n        self.enc1 = nn.Sequential(\n            nn.Conv2d(enc_in_channels, 64, kernel_size=3, padding=1),\n            ResidualBlock(64),\n        )\n        self.down1 = nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1)\n\n        # Encoder level 2\n        self.enc2 = nn.Sequential(\n            ResidualBlock(128),\n            ResidualBlock(128),\n        )\n        self.down2 = nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1)\n\n        # Encoder level 3\n        self.enc3 = nn.Sequential(\n            ResidualBlock(256),\n            ResidualBlock(256),\n        )\n        self.down3 = nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1)\n\n        # Bottleneck (512 channels at H/8, W/8)\n        self.bottleneck = nn.Sequential(\n            ResidualBlock(512),\n            ResidualBlock(512),\n        )\n\n        # Decoder level 3: upsample 512->256, concat with enc3 skip (256) = 512 channels\n        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1)\n        self.dec3 = nn.Sequential(\n            nn.Conv2d(256 + 256, 256, kernel_size=1),  # 1x1 conv: 512 -> 256\n            ResidualBlock(256),\n            ResidualBlock(256),\n        )\n\n        # Decoder level 2: upsample 256->128, concat with enc2 skip (128) = 256 channels\n        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1)\n        self.dec2 = nn.Sequential(\n            nn.Conv2d(128 + 128, 128, kernel_size=1),  # 1x1 conv: 256 -> 128\n            ResidualBlock(128),\n            ResidualBlock(128),\n        )\n\n        # Decoder level 1: upsample 128->64, concat with enc1 skip (64) = 128 channels\n        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)\n        self.dec1 = nn.Sequential(\n            nn.Conv2d(64 + 64, 64, kernel_size=1),  # 1x1 conv: 128 -> 64\n            ResidualBlock(64),\n            nn.Conv2d(64, out_channels, kernel_size=3, padding=1),\n            nn.Sigmoid(),  # Direct clean output in [0, 1]\n        )\n\n        # Hạ khởi tạo bias của tầng Conv2d cuối cùng xuống +0.2 để tránh tẩy trắng nhầm chữ bị mờ\n        last_conv = self.dec1[-2]  # Lớp nn.Conv2d(64, out_channels, kernel_size=3, padding=1)\n        if hasattr(last_conv, "bias") and last_conv.bias is not None:\n            nn.init.constant_(last_conv.bias, 0.2)\n\n    def _load_from_state_dict(\n        self, state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs\n    ):\n        """Allow seamless loading of legacy 1-channel enc1.0.weight checkpoints if needed."""\n        w_key = prefix + "enc1.0.weight"\n        if w_key in state_dict:\n            w_val = state_dict[w_key]\n            expected_shape = self.enc1[0].weight.shape\n            if w_val.ndim == 4 and w_val.shape[1] == 1 and expected_shape[1] == 2:\n                zeros = torch.zeros_like(w_val)\n                state_dict[w_key] = torch.cat([w_val, zeros], dim=1)\n        super()._load_from_state_dict(\n            state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        # Local Illumination Normalization (tương phản cục bộ giúp lộ rõ chữ trong bóng râm ShadowCast)\n        bg = F.avg_pool2d(x, kernel_size=31, stride=1, padding=15)\n        x_norm = torch.clamp(x / (bg + 1e-4), 0.0, 1.0)\n        x_in = torch.cat([x, x_norm], dim=1)  # [B, 2, H, W]\n\n        # Encoder\n        e1 = self.enc1(x_in)     # [B, 64, H, W]\n        d1 = self.down1(e1)      # [B, 128, H/2, W/2]\n\n        e2 = self.enc2(d1)       # [B, 128, H/2, W/2]\n        d2 = self.down2(e2)      # [B, 256, H/4, W/4]\n\n        e3 = self.enc3(d2)       # [B, 256, H/4, W/4]\n        d3 = self.down3(e3)      # [B, 512, H/8, W/8]\n\n        # Bottleneck\n        b = self.bottleneck(d3)  # [B, 512, H/8, W/8]\n\n        # Decoder level 3\n        u3 = self.up3(b)                 # [B, 256, H/4, W/4]\n        c3 = torch.cat([u3, e3], dim=1)  # [B, 512, H/4, W/4]\n        d_out3 = self.dec3(c3)           # [B, 256, H/4, W/4]\n\n        # Decoder level 2\n        u2 = self.up2(d_out3)            # [B, 128, H/2, W/2]\n        c2 = torch.cat([u2, e2], dim=1)  # [B, 256, H/2, W/2]\n        d_out2 = self.dec2(c2)           # [B, 128, H/2, W/2]\n\n        # Decoder level 1\n        u1 = self.up1(d_out2)            # [B, 64, H, W]\n        c1 = torch.cat([u1, e1], dim=1)  # [B, 128, H, W]\n        out = self.dec1(c1)              # [B, out_channels, H, W]\n\n        return out\n\n\ndef calculate_psnr(img1: torch.Tensor, img2: torch.Tensor) -> float:\n    """Calculate PSNR between two tensors with values in [0, 1]."""\n    mse = torch.mean((img1 - img2) ** 2).item()\n    if mse == 0:\n        return 100.0\n    return 20 * math.log10(1.0 / math.sqrt(mse))\n\n\ndef train_restoration_model(\n    train_csv: str = "datasets/metadata/train.csv",\n    val_csv: str = "datasets/metadata/val.csv",\n    output_dir: str = "models",\n    epochs: int = 20,\n    batch_size: int = 4,\n    lr: float = 1e-4,\n    image_size: int = 512,\n    manifest: str = None,\n    dataset_root: str = None,\n    resume_weights: str = "models/restoration_best.pth",\n):\n    """Train or Fine-Tune Document Restoration UNet with validation and best checkpoint saving.\n\n    If `manifest` is given (datasets/metadata/dataset_manifest.csv), train/val pairs are read from it\n    (many inputs -> one target) and `train_csv`/`val_csv` are ignored.\n    If `resume_weights` exists on disk, warm-starts the U-Net from those pretrained weights.\n    """\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    logger.info(f"Starting Document Restoration Training on device: {device}")\n\n    # Create datasets\n    img_size = (image_size, image_size)\n    if manifest:\n        train_csv = val_csv = manifest\n        train_dataset = DocumentRestorationDataset(manifest, image_size=img_size, split="train", is_train=True, root_dir=dataset_root)\n        val_dataset = DocumentRestorationDataset(manifest, image_size=img_size, split="val", is_train=False, root_dir=dataset_root)\n    else:\n        train_dataset = DocumentRestorationDataset(metadata_csv=train_csv, image_size=img_size, split="train", is_train=True)\n        val_dataset = DocumentRestorationDataset(metadata_csv=val_csv, image_size=img_size, split="val", is_train=False)\n\n        # Automatic fallback to manifest if default train_csv has no valid image pairs on disk\n        if ("datasets" in train_csv and "train.csv" in train_csv) and len(train_dataset) == 0:\n            manifest_fallback = "datasets/metadata/dataset_manifest.csv"\n            if os.path.exists(manifest_fallback):\n                logger.info(f"No valid image pairs in \'{train_csv}\', falling back to manifest: {manifest_fallback}")\n                train_dataset = DocumentRestorationDataset(manifest_fallback, image_size=img_size, split="train", is_train=True, root_dir=dataset_root)\n                val_dataset = DocumentRestorationDataset(manifest_fallback, image_size=img_size, split="val", is_train=False, root_dir=dataset_root)\n\n    if len(train_dataset) == 0:\n        logger.error(f"No training pairs found in \'{train_csv}\'. Training aborted.")\n        raise ValueError(f"No training pairs found in \'{train_csv}\'. Training aborted.")\n    if len(val_dataset) == 0:\n        logger.warning(f"No validation pairs found in \'{val_csv}\'. Will train without validation.")\n\n    logger.info(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")\n\n    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)\n    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)\n\n    # Model\n    model = DocumentRestorationUNet().to(device)\n    total_params = sum(p.numel() for p in model.parameters())\n    logger.info(f"Model parameters: {total_params:,}")\n\n    if resume_weights and os.path.exists(resume_weights):\n        state_dict = torch.load(resume_weights, map_location=device, weights_only=True)\n        model.load_state_dict(state_dict)\n        logger.info(f"Warm-Start (Fine-Tune): Loaded pretrained weights from \'{resume_weights}\'")\n\n    # Loss and optimizer: Multi-purpose DocumentCleanLoss (Weighted Charbonnier + BCE + Sobel + SSIM + Bimodal + BleedThroughSuppression)\n    criterion = DocumentCleanLoss().to(device)\n    optimizer = optim.AdamW(model.parameters(), lr=lr)\n\n    # Checkpoint directory\n    os.makedirs(output_dir, exist_ok=True)\n    best_model_path = os.path.join(output_dir, "restoration_best.pth")\n    last_model_path = os.path.join(output_dir, "restoration_last.pth")\n\n    best_val_psnr = -float("inf")\n    training_log = []\n\n    for epoch in range(1, epochs + 1):\n        # ======================== TRAIN ========================\n        model.train()\n        train_loss = 0.0\n        train_psnr = 0.0\n        train_steps = 0\n\n        for deg, clean in train_loader:\n            deg, clean = deg.to(device), clean.to(device)\n            optimizer.zero_grad()\n\n            pred = model(deg)\n            loss = criterion(pred, clean)\n            loss.backward()\n            optimizer.step()\n\n            train_loss += loss.item()\n            with torch.no_grad():\n                train_psnr += calculate_psnr(pred, clean)\n            train_steps += 1\n\n        avg_train_loss = train_loss / max(train_steps, 1)\n        avg_train_psnr = train_psnr / max(train_steps, 1)\n\n        # ======================== VALIDATION ========================\n        avg_val_loss = 0.0\n        avg_val_psnr = 0.0\n\n        if len(val_dataset) > 0:\n            model.eval()\n            val_loss = 0.0\n            val_psnr = 0.0\n            val_steps = 0\n\n            with torch.no_grad():\n                for deg, clean in val_loader:\n                    deg, clean = deg.to(device), clean.to(device)\n                    pred = model(deg)\n                    loss = criterion(pred, clean)\n                    val_loss += loss.item()\n                    val_psnr += calculate_psnr(pred, clean)\n                    val_steps += 1\n\n            avg_val_loss = val_loss / max(val_steps, 1)\n            avg_val_psnr = val_psnr / max(val_steps, 1)\n\n        # Log\n        epoch_info = {\n            "epoch": epoch,\n            "train_loss": round(avg_train_loss, 6),\n            "train_psnr": round(avg_train_psnr, 2),\n            "val_loss": round(avg_val_loss, 6),\n            "val_psnr": round(avg_val_psnr, 2),\n        }\n        training_log.append(epoch_info)\n\n        logger.info(\n            f"Epoch [{epoch}/{epochs}] "\n            f"Train Loss: {avg_train_loss:.4f} | Train PSNR: {avg_train_psnr:.2f} dB | "\n            f"Val Loss: {avg_val_loss:.4f} | Val PSNR: {avg_val_psnr:.2f} dB"\n        )\n\n        # Save last checkpoint\n        torch.save(model.state_dict(), last_model_path)\n\n        # Save best checkpoint based on validation PSNR\n        if avg_val_psnr > best_val_psnr and len(val_dataset) > 0:\n            best_val_psnr = avg_val_psnr\n            torch.save(model.state_dict(), best_model_path)\n            logger.info(f"  -> New best model saved (Val PSNR: {best_val_psnr:.2f} dB)")\n\n    # If no validation data, save last as best\n    if len(val_dataset) == 0:\n        torch.save(model.state_dict(), best_model_path)\n        logger.info("No validation data. Saved last model as best.")\n\n    # Save training log\n    log_path = os.path.join(output_dir, "training_log.json")\n    val_psnr_record = round(best_val_psnr, 2) if best_val_psnr != -float("inf") else None\n    with open(log_path, "w", encoding="utf-8") as f:\n        json.dump({\n            "device": str(device),\n            "epochs": epochs,\n            "batch_size": batch_size,\n            "learning_rate": lr,\n            "image_size": image_size,\n            "train_samples": len(train_dataset),\n            "val_samples": len(val_dataset),\n            "best_val_psnr": val_psnr_record,\n            "log": training_log,\n        }, f, indent=2)\n\n    logger.info(f"Training complete. Best Val PSNR: {val_psnr_record if val_psnr_record is not None else \'N/A\'} dB")\n    logger.info(f"Best model: {best_model_path}")\n    logger.info(f"Last model: {last_model_path}")\n    logger.info(f"Training log: {log_path}")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="Train Document Image Restoration UNet")\n    parser.add_argument("--train-csv", default="datasets/metadata/train.csv",\n                        help="Path to training metadata CSV")\n    parser.add_argument("--val-csv", default="datasets/metadata/val.csv",\n                        help="Path to validation metadata CSV")\n    parser.add_argument("--output-dir", default="models",\n                        help="Directory to save model checkpoints")\n    parser.add_argument("--epochs", type=int, default=20,\n                        help="Number of training epochs")\n    parser.add_argument("--batch-size", type=int, default=4,\n                        help="Batch size for training")\n    parser.add_argument("--lr", type=float, default=1e-4,\n                        help="Learning rate")\n    parser.add_argument("--image-size", type=int, default=512,\n                        help="Target image size (square)")\n    parser.add_argument("--manifest", default=None,\n                        help="dataset_manifest.csv (many inputs -> one target); overrides --train-csv/--val-csv")\n    parser.add_argument("--dataset-root", default=None,\n                        help="Root that manifest paths are relative to (default: datasets/)")\n    parser.add_argument("--resume-weights", default="models/restoration_best.pth",\n                        help="Path to pretrained weights for warm-start fine-tuning")\n    args = parser.parse_args()\n\n    try:\n        train_restoration_model(\n            train_csv=args.train_csv,\n            val_csv=args.val_csv,\n            output_dir=args.output_dir,\n            epochs=args.epochs,\n            batch_size=args.batch_size,\n            lr=args.lr,\n            image_size=args.image_size,\n            manifest=args.manifest,\n            dataset_root=args.dataset_root,\n            resume_weights=args.resume_weights,\n        )\n    except Exception as e:\n        logger.error(f"Training failed: {e}")\n        sys.exit(1)\n'

EVALUATE_RESTORATION_CODE = 'import os\nimport sys\nimport argparse\nimport csv\nimport cv2\nimport numpy as np\nimport torch\nimport math\nimport json\nfrom PIL import Image\n\n# Add parent directory to path to import training modules\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), \'..\')))\nfrom training.train_restoration import DocumentRestorationUNet\nfrom training.restoration_dataset import (\n    load_image,\n    save_image,\n    unsharp_mask,\n    resize_with_padding,\n    remove_padding,\n    normalize_clean_target,\n    clean_target_normalization,\n    infer_single_image_tiled,\n    GEOMETRIC_DISTORTION_KEYWORDS,\n)\n\n\ndef calculate_psnr(img1: np.ndarray, img2: np.ndarray) -> float:\n    """Calculate PSNR between two uint8 images."""\n    mse = np.mean((img1.astype(float) - img2.astype(float)) ** 2)\n    if mse == 0:\n        return 100.0\n    return 20 * math.log10(255.0 / math.sqrt(mse))\n\n\ndef calculate_ssim(img1: np.ndarray, img2: np.ndarray) -> float:\n    """Calculate SSIM between two uint8 images."""\n    c1 = (0.01 * 255) ** 2\n    c2 = (0.03 * 255) ** 2\n    img1 = img1.astype(np.float64)\n    img2 = img2.astype(np.float64)\n    mu1 = cv2.GaussianBlur(img1, (11, 11), 1.5)\n    mu2 = cv2.GaussianBlur(img2, (11, 11), 1.5)\n    mu1_sq = mu1 ** 2\n    mu2_sq = mu2 ** 2\n    mu1_mu2 = mu1 * mu2\n    sigma1_sq = cv2.GaussianBlur(img1 ** 2, (11, 11), 1.5) - mu1_sq\n    sigma2_sq = cv2.GaussianBlur(img2 ** 2, (11, 11), 1.5) - mu2_sq\n    sigma12 = cv2.GaussianBlur(img1 * img2, (11, 11), 1.5) - mu1_mu2\n    ssim_map = ((2 * mu1_mu2 + c1) * (2 * sigma12 + c2)) / (\n        (mu1_sq + mu2_sq + c1) * (sigma1_sq + sigma2_sq + c2)\n    )\n    return float(np.mean(ssim_map))\n\n\ndef _load_pairs_from_csv(csv_path: str, default_split: str = "test"):\n    """Load valid image pairs and unpaired blind evaluation entries from a split CSV (test.csv or val.csv)."""\n    pairs = []\n    unpaired = []\n    if not csv_path or not os.path.exists(csv_path):\n        return pairs, unpaired\n    with open(csv_path, \'r\', encoding=\'utf-8-sig\') as f:\n        reader = csv.DictReader(f)\n        for row in reader:\n            raw_in = (row.get(\'input\') or \'\').strip()\n            raw_tgt = (row.get(\'target\') or \'\').strip()\n            in_p = os.path.normpath(raw_in) if raw_in else \'\'\n            tgt_p = os.path.normpath(raw_tgt) if raw_tgt else \'\'\n            deg_str = (row.get(\'degradation\') or row.get(\'blur_level\') or \'\').strip().lower()\n            in_stem = os.path.splitext(os.path.basename(in_p))[0].lower() if in_p else \'\'\n\n            is_blind = (\n                in_stem in (\'real_11\', \'real_12\')\n                or deg_str in (\'unpaired_real\', \'blind_real\')\n                or (raw_tgt == \'\' and bool(raw_in))\n            )\n            if is_blind:\n                if in_p and os.path.exists(in_p):\n                    row_copy = dict(row)\n                    row_copy[\'input\'] = in_p\n                    row_copy[\'target\'] = \'\'\n                    row_copy[\'split\'] = row.get(\'split\', default_split) or default_split\n                    row_copy[\'is_unpaired\'] = True\n                    unpaired.append(row_copy)\n                continue\n\n            if in_p and tgt_p and os.path.exists(in_p) and os.path.exists(tgt_p):\n                row_copy = dict(row)\n                row_copy[\'input\'] = in_p\n                row_copy[\'target\'] = tgt_p\n                row_copy[\'split\'] = row.get(\'split\', default_split) or default_split\n                row_copy[\'is_unpaired\'] = False\n                pairs.append(row_copy)\n    return pairs, unpaired\n\n\ndef _collect_blind_real_samples(existing_unpaired):\n    """Ensure datasets/processed/test/input/real_*.jpg and datasets/processed/val/input/real_*.jpg are included."""\n    seen_paths = {os.path.abspath(item[\'input\']) for item in existing_unpaired if item.get(\'input\')}\n    blind_list = list(existing_unpaired)\n\n    candidates = [\n        (\'test\', \'datasets/processed/test/input/real_11.jpg\'),\n        (\'val\', \'datasets/processed/val/input/real_12.jpg\'),\n    ]\n    for split_name, folder in [(\'test\', \'datasets/processed/test/input\'), (\'val\', \'datasets/processed/val/input\')]:\n        if os.path.exists(folder):\n            for fn in sorted(os.listdir(folder)):\n                if fn.lower().startswith(\'real_\') and os.path.splitext(fn)[1].lower() in (\'.jpg\', \'.jpeg\', \'.png\', \'.webp\'):\n                    candidates.append((split_name, os.path.join(folder, fn)))\n\n    for split_name, rel_p in candidates:\n        norm_p = os.path.normpath(rel_p)\n        if os.path.exists(norm_p):\n            abs_p = os.path.abspath(norm_p)\n            if abs_p not in seen_paths:\n                seen_paths.add(abs_p)\n                stem = os.path.splitext(os.path.basename(norm_p))[0]\n                blind_list.append({\n                    \'input\': norm_p,\n                    \'target\': \'\',\n                    \'source_id\': stem,\n                    \'degradation\': \'unpaired_real\',\n                    \'blur_level\': f\'real_blind_{split_name}\',\n                    \'split\': split_name,\n                    \'is_unpaired\': True,\n                })\n    return blind_list\n\n\ndef create_two_column_blind_comparison(raw_bgr: np.ndarray, pure_unet_gray: np.ndarray, target_h: int = 680) -> np.ndarray:\n    """\n    Create an honest 2-column comparison sheet for Unpaired Real Evaluation:\n    [1. Anh Real Goc (Input duy nhat) | 2. Ket qua 100% Pure U-Net (Output tu AI)]\n    """\n    if raw_bgr.ndim == 2:\n        raw_bgr = cv2.cvtColor(raw_bgr, cv2.COLOR_GRAY2BGR)\n    if pure_unet_gray.ndim == 2:\n        unet_bgr = cv2.cvtColor(pure_unet_gray, cv2.COLOR_GRAY2BGR)\n    else:\n        unet_bgr = pure_unet_gray.copy()\n\n    def _resize_col(img: np.ndarray, th: int) -> np.ndarray:\n        hi, wi = img.shape[:2]\n        scale = th / float(max(hi, 1))\n        nw = max(240, int(wi * scale))\n        return cv2.resize(img, (nw, th), interpolation=cv2.INTER_AREA)\n\n    def _add_header(img: np.ndarray, title: str, color: tuple) -> np.ndarray:\n        header_h = 46\n        canvas = np.full((img.shape[0] + header_h, img.shape[1], 3), 245, dtype=np.uint8)\n        canvas[header_h:, :] = img\n        cv2.putText(canvas, title, (12, 31), cv2.FONT_HERSHEY_SIMPLEX, 0.62, color, 2, cv2.LINE_AA)\n        return canvas\n\n    col1 = _add_header(\n        _resize_col(raw_bgr, target_h),\n        "1. Anh Real Goc (Input duy nhat)",\n        (0, 0, 210),\n    )\n    col2 = _add_header(\n        _resize_col(unet_bgr, target_h),\n        "2. Ket qua 100% Pure U-Net (Output tu AI)",\n        (180, 60, 0),\n    )\n    divider = np.full((col1.shape[0], 4, 3), 200, dtype=np.uint8)\n    return np.hstack([col1, divider, col2])\n\n\ndef _summarize_group(items):\n    """Compute average PSNR/SSIM summary for a subset of evaluated samples."""\n    if not items:\n        return {\n            \'count\': 0,\n            \'restored_psnr\': 0.0,\n            \'blur_psnr\': 0.0,\n            \'psnr_improvement\': 0.0,\n            \'restored_ssim\': 0.0,\n            \'blur_ssim\': 0.0,\n            \'ssim_improvement\': 0.0,\n        }\n    r_psnr = float(np.mean([m[\'restored_psnr\'] for m in items]))\n    b_psnr = float(np.mean([m[\'blur_psnr\'] for m in items]))\n    r_ssim = float(np.mean([m[\'restored_ssim\'] for m in items]))\n    b_ssim = float(np.mean([m[\'blur_ssim\'] for m in items]))\n    return {\n        \'count\': len(items),\n        \'restored_psnr\': round(r_psnr, 2),\n        \'blur_psnr\': round(b_psnr, 2),\n        \'psnr_improvement\': round(r_psnr - b_psnr, 2),\n        \'restored_ssim\': round(r_ssim, 4),\n        \'blur_ssim\': round(b_ssim, 4),\n        \'ssim_improvement\': round(r_ssim - b_ssim, 4),\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Evaluate Document Restoration AI Model")\n    parser.add_argument(\'--model\', type=str, default=\'models/restoration_best.pth\',\n                        help=\'Path to model checkpoint\')\n    parser.add_argument(\'--test-csv\', type=str, default=\'datasets/metadata/test.csv\',\n                        help=\'Path to test split CSV\')\n    parser.add_argument(\'--val-csv\', type=str, default=\'datasets/metadata/val.csv\',\n                        help=\'Path to val split CSV\')\n    parser.add_argument(\'--include-val\', action=\'store_true\', default=True,\n                        help=\'Also export comparison sheets for val.csv\')\n    parser.add_argument(\'--no-val\', action=\'store_false\', dest=\'include_val\',\n                        help=\'Evaluate only test.csv without val.csv\')\n    parser.add_argument(\'--image-size\', type=int, default=512,\n                        help=\'Target model tile size for sliding window inference\')\n    parser.add_argument(\'--overlap\', type=int, default=128,\n                        help=\'Overlap between sliding window tiles\')\n    parser.add_argument(\'--blind-out-dir\', type=str, default=\'outputs/restoration/real_blind_test\',\n                        help=\'Output directory for 100% Pure U-Net blind test images (real_11.jpg & real_12.jpg)\')\n    args = parser.parse_args()\n\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    print(f"Device: {device}")\n\n    # Load Model\n    model = DocumentRestorationUNet().to(device)\n\n    if os.path.exists(args.model):\n        model.load_state_dict(torch.load(args.model, map_location=device, weights_only=True))\n        print(f"Loaded model from {args.model}")\n    else:\n        print(f"ERROR: Model not found at {args.model}.")\n        return\n\n    model.eval()\n\n    output_dir = \'outputs/restoration/test\'\n    comp_dir = \'outputs/restoration/comparison\'\n    blind_dir = args.blind_out_dir\n    os.makedirs(output_dir, exist_ok=True)\n    os.makedirs(comp_dir, exist_ok=True)\n    os.makedirs(blind_dir, exist_ok=True)\n\n    # Remove old comparison images so only current evaluation results exist\n    for old_f in os.listdir(comp_dir):\n        if old_f.endswith("_comparison.png"):\n            try:\n                os.remove(os.path.join(comp_dir, old_f))\n            except OSError:\n                pass\n\n    # Read test CSV and val CSV\n    test_pairs, test_unpaired = _load_pairs_from_csv(args.test_csv, default_split="test")\n    val_pairs, val_unpaired = _load_pairs_from_csv(args.val_csv, default_split="val") if args.include_val else ([], [])\n\n    eval_pairs = test_pairs + val_pairs\n    blind_samples = _collect_blind_real_samples(test_unpaired + val_unpaired)\n\n    if len(eval_pairs) == 0:\n        manifest_p = "datasets/metadata/dataset_manifest.csv"\n        if os.path.exists(manifest_p):\n            print(f"No valid files in \'{args.test_csv}\'. Loading evaluation pairs from \'{manifest_p}\'...")\n            with open(manifest_p, \'r\', encoding=\'utf-8-sig\') as f:\n                reader = csv.DictReader(f)\n                for r in reader:\n                    if r.get("split", "").strip() not in ("test", "val"):\n                        continue\n                    raw_in = r.get("input_path", "").strip()\n                    raw_tgt = r.get("target_path", "").strip()\n                    if not raw_in or not raw_tgt:\n                        continue\n                    in_p = os.path.normpath(os.path.join("datasets", raw_in))\n                    tgt_p = os.path.normpath(os.path.join("datasets", raw_tgt))\n                    if os.path.exists(in_p) and os.path.exists(tgt_p):\n                        eval_pairs.append({\n                            \'input\': in_p,\n                            \'target\': tgt_p,\n                            \'source_id\': r.get(\'original_id\', \'doc\'),\n                            \'blur_level\': r.get(\'degradation_type\', \'degraded\') + \'_\' + r.get(\'variant_id\', \'01\'),\n                            \'split\': r.get(\'split\', \'test\'),\n                        })\n        elif len(blind_samples) == 0:\n            print("ERROR: No valid test pairs found.")\n            return\n\n    print(\n        f"Total paired samples to evaluate: {len(eval_pairs)} (test={len(test_pairs)}, val={len(val_pairs)}) | "\n        f"Unpaired Blind Real samples: {len(blind_samples)}"\n    )\n    all_metrics = []\n\n    for pair in eval_pairs:\n        blur_path = os.path.normpath(pair[\'input\'].strip())\n        orig_path = os.path.normpath(pair[\'target\'].strip())\n        source_id = pair.get(\'source_id\') or pair.get(\'original_id\') or \'unknown\'\n        blur_level = pair.get(\'blur_level\') or pair.get(\'degradation\') or os.path.splitext(os.path.basename(blur_path))[0]\n        split_tag = pair.get(\'split\', \'test\')\n\n        blur_img = load_image(blur_path)\n        orig_img = load_image(orig_path)\n        if blur_img is None or orig_img is None:\n            print(f"  SKIP {source_id} ({blur_level}): failed to load images")\n            continue\n\n        # Convert to Grayscale and apply synchronized clean_target_normalization on Ground Truth\n        blur_gray = cv2.cvtColor(blur_img, cv2.COLOR_BGR2GRAY)\n        orig_gray = cv2.cvtColor(orig_img, cv2.COLOR_BGR2GRAY)\n        orig_gray = clean_target_normalization(orig_gray)\n\n        # Align dimensions if input and clean target differ in resolution (e.g. LongestMaxSize)\n        # Always resize degraded input to match clean target resolution\n        if blur_gray.shape[:2] != orig_gray.shape[:2]:\n            blur_gray = cv2.resize(blur_gray, (orig_gray.shape[1], orig_gray.shape[0]), interpolation=cv2.INTER_CUBIC)\n\n        # Sliding Window Tiling Inference (512x512 with 128 overlap & Gaussian feather blending)\n        restored_cropped = infer_single_image_tiled(\n            model, blur_gray, device, tile_size=args.image_size, overlap=args.overlap\n        )\n\n        # Post-processing: Unsharp Masking\n        restored_final = unsharp_mask(restored_cropped, amount=0.8, radius=1.0)\n\n        # Compute metrics on grayscale images\n        l1 = float(np.mean(np.abs(orig_gray.astype(float) - restored_final.astype(float))))\n        mse = float(np.mean((orig_gray.astype(float) - restored_final.astype(float)) ** 2))\n        psnr = calculate_psnr(orig_gray, restored_final)\n        ssim = calculate_ssim(orig_gray, restored_final)\n\n        # Baseline metrics (degraded vs normalized clean GT)\n        blur_psnr = calculate_psnr(orig_gray, blur_gray)\n        blur_ssim = calculate_ssim(orig_gray, blur_gray)\n\n        check_text = f"{os.path.basename(blur_path)} {blur_level}".lower()\n        is_geometric = any(kw in check_text for kw in GEOMETRIC_DISTORTION_KEYWORDS)\n\n        metrics = {\n            \'split\': split_tag,\n            \'source_id\': source_id,\n            \'blur_level\': blur_level,\n            \'is_geometric\': is_geometric,\n            \'restored_l1\': round(l1, 4),\n            \'restored_mse\': round(mse, 4),\n            \'restored_psnr\': round(psnr, 2),\n            \'restored_ssim\': round(ssim, 4),\n            \'blur_psnr\': round(blur_psnr, 2),\n            \'blur_ssim\': round(blur_ssim, 4),\n            \'psnr_improvement\': round(psnr - blur_psnr, 2),\n        }\n        all_metrics.append(metrics)\n        geom_tag = "GEOM" if is_geometric else "ALIGNED"\n        print(f"  [{split_tag.upper()}:{geom_tag} | {source_id}/{blur_level}] "\n              f"Restored PSNR: {psnr:.2f} dB (Degraded: {blur_psnr:.2f} dB, diff={psnr-blur_psnr:+.2f} dB) | "\n              f"SSIM: {ssim:.4f} (Degraded: {blur_ssim:.4f})")\n\n        # Save images in structured folders\n        sample_name = f"{source_id}_{blur_level}"\n        save_dir = os.path.join(output_dir, sample_name)\n        os.makedirs(save_dir, exist_ok=True)\n        cv2.imwrite(os.path.join(save_dir, \'input_blur.png\'), blur_gray)\n        cv2.imwrite(os.path.join(save_dir, \'restored.png\'), restored_final)\n        cv2.imwrite(os.path.join(save_dir, \'original.png\'), orig_gray)\n\n        # Create comparison contact sheet: Original (GT) | Degraded Input | U-Net Restored\n        target_h = 450\n        def resize_for_display(img, th):\n            hi, wi = img.shape[:2]\n            scale = th / hi\n            return cv2.resize(img, (int(wi * scale), th), interpolation=cv2.INTER_AREA)\n\n        def add_label(img, label):\n            img_c = img.copy()\n            cv2.putText(img_c, label, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)\n            return img_c\n\n        disp_orig = add_label(resize_for_display(cv2.cvtColor(orig_gray, cv2.COLOR_GRAY2BGR), target_h), "1. Ground Truth")\n        disp_blur = add_label(resize_for_display(cv2.cvtColor(blur_gray, cv2.COLOR_GRAY2BGR), target_h), "2. Degraded Input")\n        disp_rest = add_label(resize_for_display(cv2.cvtColor(restored_final, cv2.COLOR_GRAY2BGR), target_h), "3. U-Net Restored")\n\n        contact_sheet = np.hstack([disp_orig, disp_blur, disp_rest])\n        comparison_path = os.path.join(comp_dir, f"{sample_name}_comparison.png")\n        cv2.imwrite(comparison_path, contact_sheet)\n        print(f"     -> Saved comparison contact sheet: {comparison_path}")\n\n    # ======================== UNPAIRED REAL BLIND EVALUATION (100% PURE U-NET) ========================\n    blind_results = []\n    if blind_samples:\n        print("\\n" + "=" * 68)\n        print("UNPAIRED REAL BLIND EVALUATION (100% PURE U-NET INFERENCE - NO TARGET)")\n        print("=" * 68)\n        for b_item in blind_samples:\n            raw_path = os.path.normpath(b_item[\'input\'])\n            split_tag = b_item.get(\'split\', \'test\')\n            stem = os.path.splitext(os.path.basename(raw_path))[0]\n            raw_bgr = load_image(raw_path)\n            if raw_bgr is None:\n                print(f"  [BLIND-{split_tag.upper()}] SKIP {raw_path}: failed to load")\n                continue\n\n            raw_gray = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)\n\n            # 100% Pure U-Net Inference directly from raw_gray (NO post-processing, NO unsharp mask, NO fake target)\n            pure_unet_gray = infer_single_image_tiled(\n                model, raw_gray, device, tile_size=args.image_size, overlap=args.overlap, refine_glyphs=False\n            )\n\n            pure_out_path = os.path.join(blind_dir, f"{stem}_pure_unet.png")\n            comp_2col_path = os.path.join(blind_dir, f"{stem}_blind_comparison.png")\n\n            two_col_sheet = create_two_column_blind_comparison(raw_bgr, pure_unet_gray, target_h=680)\n            save_image(pure_out_path, pure_unet_gray)\n            save_image(comp_2col_path, two_col_sheet)\n\n            blind_rec = {\n                \'split\': split_tag,\n                \'source_id\': stem,\n                \'input_path\': raw_path,\n                \'pure_unet_output_path\': pure_out_path,\n                \'comparison_2col_path\': comp_2col_path,\n                \'mode\': \'100% Pure U-Net (Unpaired Blind Evaluation - No Target)\',\n            }\n            blind_results.append(blind_rec)\n            print(\n                f"  [BLIND-{split_tag.upper()} | {stem}] 100% Pure U-Net Output -> {pure_out_path} | "\n                f"2-Column Comparison -> {comp_2col_path}"\n            )\n\n    # Summary separated into pixel_aligned_metrics and geometric_distorted_metrics\n    if all_metrics or blind_results:\n        test_only = [m for m in all_metrics if m.get(\'split\') == \'test\'] or all_metrics\n        pixel_aligned = [m for m in test_only if not m.get(\'is_geometric\', False)]\n        geom_distorted = [m for m in test_only if m.get(\'is_geometric\', False)]\n\n        overall_summary = _summarize_group(test_only)\n        aligned_summary = _summarize_group(pixel_aligned)\n        geom_summary = _summarize_group(geom_distorted)\n\n        print("\\n" + "=" * 68)\n        print("TEST RESULTS SUMMARY (Separated by Alignment)")\n        print("=" * 68)\n        print(f"1. Pixel-Aligned Samples ({aligned_summary[\'count\']} samples - True Pixel Restoration):")\n        print(f"   Avg Restored PSNR: {aligned_summary[\'restored_psnr\']:.2f} dB (Blur: {aligned_summary[\'blur_psnr\']:.2f} dB, Diff: {aligned_summary[\'psnr_improvement\']:+.2f} dB)")\n        print(f"   Avg Restored SSIM: {aligned_summary[\'restored_ssim\']:.4f} (Blur: {aligned_summary[\'blur_ssim\']:.4f}, Diff: {aligned_summary[\'ssim_improvement\']:+.4f})")\n        print("-" * 68)\n        print(f"2. Geometric-Distorted Samples ({geom_summary[\'count\']} samples - Perspective/BookBinding/Optical):")\n        print(f"   Avg Restored PSNR: {geom_summary[\'restored_psnr\']:.2f} dB (Blur: {geom_summary[\'blur_psnr\']:.2f} dB, Diff: {geom_summary[\'psnr_improvement\']:+.2f} dB)")\n        print(f"   Avg Restored SSIM: {geom_summary[\'restored_ssim\']:.4f} (Blur: {geom_summary[\'blur_ssim\']:.4f}, Diff: {geom_summary[\'ssim_improvement\']:+.4f})")\n        print("-" * 68)\n        print(f"3. Overall Paired Test Samples ({overall_summary[\'count\']} test samples, {len(all_metrics)} total incl. val):")\n        print(f"   Avg Restored PSNR: {overall_summary[\'restored_psnr\']:.2f} dB | Avg Restored SSIM: {overall_summary[\'restored_ssim\']:.4f}")\n        print("-" * 68)\n        print(f"4. Unpaired Real Blind Evaluation ({len(blind_results)} samples - 100% Pure U-Net, No Target): {[b[\'source_id\'] for b in blind_results]}")\n        print("=" * 68)\n\n        metrics_path = \'outputs/restoration/test_metrics.json\'\n        os.makedirs(os.path.dirname(metrics_path), exist_ok=True)\n        with open(metrics_path, \'w\', encoding=\'utf-8\') as f:\n            json.dump({\n                \'pixel_aligned_metrics\': aligned_summary,\n                \'geometric_distorted_metrics\': geom_summary,\n                \'average\': overall_summary,\n                \'per_image\': all_metrics,\n                \'blind_real_samples\': blind_results,\n                \'total_samples\': len(all_metrics),\n            }, f, indent=2, ensure_ascii=False)\n        print(f"Metrics saved to: {metrics_path}")\n\n\nif __name__ == \'__main__\':\n    main()\n\n\n'

INFER_RESTORATION_CODE = 'import os\nimport sys\nimport argparse\nimport glob\nimport cv2\nimport numpy as np\nimport torch\n\n# Add parent directory to path to import training modules\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))\nfrom training.train_restoration import DocumentRestorationUNet\nfrom training.restoration_dataset import (\n    load_image,\n    save_image,\n    unsharp_mask,\n    resize_with_padding,\n    remove_padding,\n    infer_single_image_tiled,\n)\n\n\ndef restore_image_for_yolov8(\n    model: torch.nn.Module,\n    img_bgr: np.ndarray,\n    device: torch.device,\n    tile_size: int = 512,\n    overlap: int = 128,\n    preserve_color: bool = False,\n    refine_glyphs: bool = False,\n    apply_postprocess: bool = False,\n) -> np.ndarray:\n    """\n    AI-Centric Document Restoration for YOLOv8:\n    1. Passes image directly through `DocumentRestorationUNet` (`infer_single_image_tiled`).\n    2. Default mode (preserve_color=False, apply_postprocess=False):\n       Output is 100% Pure U-Net output tensor converted to BGR uint8,\n       WITHOUT any background clipping (e.g. np.where(y < 245, ..., 255)) or unsharp masking.\n    3. If preserve_color=True: merges U-Net\'s raw prediction with original chrominance (Cr, Cb).\n    4. If apply_postprocess=True: optionally applies stroke unsharp masking and white floor clipping.\n    """\n    if img_bgr.ndim == 2:\n        gray = img_bgr\n        img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)\n    else:\n        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)\n\n    # 100% Pure U-Net forward pass\n    y_restored = infer_single_image_tiled(\n        model, gray, device, tile_size=tile_size, overlap=overlap, refine_glyphs=refine_glyphs\n    )\n\n    if apply_postprocess:\n        y_proc = np.where(y_restored < 245, unsharp_mask(y_restored, amount=0.65, radius=1.0), 255).astype(np.uint8)\n    else:\n        y_proc = y_restored\n\n    if not preserve_color:\n        return cv2.cvtColor(y_proc, cv2.COLOR_GRAY2BGR)\n\n    ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)\n    _, cr_chan, cb_chan = cv2.split(ycrcb)\n    chroma_sat = np.hypot(cr_chan.astype(np.float32) - 128.0, cb_chan.astype(np.float32) - 128.0)\n    color_fg_mask = (y_proc < 245) & (chroma_sat > 18.0)\n    cr_clean = np.where(color_fg_mask, cr_chan, 128).astype(np.uint8)\n    cb_clean = np.where(color_fg_mask, cb_chan, 128).astype(np.uint8)\n    return cv2.cvtColor(cv2.merge([y_proc, cr_clean, cb_clean]), cv2.COLOR_YCrCb2BGR)\n\n\ndef create_three_column_preview(\n    raw_bgr: np.ndarray,\n    target_gray_or_bgr: np.ndarray,\n    unet_bgr: np.ndarray,\n    display_height: int = 680,\n) -> np.ndarray:\n    """\n    Build a 3-column comparison sheet:\n    [1. Anh Real Goc (Input) | 2. Dap an Mau (Target) | 3. Ket qua tu Model U-Net (Output cho YOLO)]\n    """\n    if raw_bgr.ndim == 2:\n        raw_bgr = cv2.cvtColor(raw_bgr, cv2.COLOR_GRAY2BGR)\n    if target_gray_or_bgr.ndim == 2:\n        target_bgr = cv2.cvtColor(target_gray_or_bgr, cv2.COLOR_GRAY2BGR)\n    else:\n        target_bgr = target_gray_or_bgr.copy()\n    if unet_bgr.ndim == 2:\n        unet_bgr = cv2.cvtColor(unet_bgr, cv2.COLOR_GRAY2BGR)\n\n    def _resize_col(img: np.ndarray, th: int) -> np.ndarray:\n        h, w = img.shape[:2]\n        scale = th / float(max(h, 1))\n        nw = max(220, int(w * scale))\n        return cv2.resize(img, (nw, th), interpolation=cv2.INTER_AREA)\n\n    def _add_header(img: np.ndarray, title: str, color: tuple) -> np.ndarray:\n        header_h = 44\n        canvas = np.full((img.shape[0] + header_h, img.shape[1], 3), 245, dtype=np.uint8)\n        canvas[header_h:, :] = img\n        cv2.putText(canvas, title, (10, 29), cv2.FONT_HERSHEY_SIMPLEX, 0.58, color, 2, cv2.LINE_AA)\n        return canvas\n\n    col1 = _add_header(_resize_col(raw_bgr, display_height), "1. Anh Real Goc (Input)", (0, 0, 210))\n    col2 = _add_header(_resize_col(target_bgr, display_height), "2. Dap an Mau (Target)", (0, 140, 0))\n    col3 = _add_header(_resize_col(unet_bgr, display_height), "3. Ket qua tu Model U-Net (Output cho YOLO)", (180, 60, 0))\n\n    # Match column widths if needed\n    divider = np.full((col1.shape[0], 4, 3), 200, dtype=np.uint8)\n    return np.hstack([col1, divider, col2, divider, col3])\n\n\ndef create_two_column_blind_preview(\n    raw_bgr: np.ndarray,\n    unet_bgr: np.ndarray,\n    display_height: int = 680,\n) -> np.ndarray:\n    """\n    Build an honest 2-column blind evaluation sheet for unpaired real images (e.g., real_11.jpg, real_12.jpg):\n    [1. Anh Real Goc (Input duy nhat) | 2. Ket qua 100% Pure U-Net (Output tu AI)]\n    """\n    if raw_bgr.ndim == 2:\n        raw_bgr = cv2.cvtColor(raw_bgr, cv2.COLOR_GRAY2BGR)\n    if unet_bgr.ndim == 2:\n        unet_bgr = cv2.cvtColor(unet_bgr, cv2.COLOR_GRAY2BGR)\n\n    def _resize_col(img: np.ndarray, th: int) -> np.ndarray:\n        h, w = img.shape[:2]\n        scale = th / float(max(h, 1))\n        nw = max(240, int(w * scale))\n        return cv2.resize(img, (nw, th), interpolation=cv2.INTER_AREA)\n\n    def _add_header(img: np.ndarray, title: str, color: tuple) -> np.ndarray:\n        header_h = 44\n        canvas = np.full((img.shape[0] + header_h, img.shape[1], 3), 245, dtype=np.uint8)\n        canvas[header_h:, :] = img\n        cv2.putText(canvas, title, (10, 29), cv2.FONT_HERSHEY_SIMPLEX, 0.58, color, 2, cv2.LINE_AA)\n        return canvas\n\n    col1 = _add_header(\n        _resize_col(raw_bgr, display_height),\n        "1. Anh Real Goc (Input duy nhat)",\n        (0, 0, 210),\n    )\n    col2 = _add_header(\n        _resize_col(unet_bgr, display_height),\n        "2. Ket qua 100% Pure U-Net (Output tu AI)",\n        (180, 60, 0),\n    )\n\n    divider = np.full((col1.shape[0], 4, 3), 200, dtype=np.uint8)\n    return np.hstack([col1, divider, col2])\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Run DocumentRestorationUNet Inference for YOLOv8")\n    parser.add_argument("--model", type=str, default="models/restoration_best.pth")\n    parser.add_argument("--input-dir", "--input", dest="input_dir", type=str, default="datasets/real")\n    parser.add_argument("--target-dir", type=str, default="datasets/processed/real_pairs/target")\n    parser.add_argument("--output-dir", "--output", dest="output_dir", type=str, default="outputs/restoration_real")\n    parser.add_argument("--preview-dir", type=str, default="datasets/real/unet_results_preview")\n    parser.add_argument("--blind-out-dir", type=str, default="outputs/restoration/real_blind_test")\n    parser.add_argument("--tile-size", type=int, default=512, help="Sliding window tile size")\n    parser.add_argument("--overlap", type=int, default=128, help="Overlap between tiles")\n    parser.add_argument(\n        "--preserve-color",\n        action="store_true",\n        default=False,\n        help="Preserve foreground color via YCrCb space while restoring luminance Y with U-Net (default: False for Pure U-Net Grayscale)",\n    )\n    parser.add_argument(\n        "--postprocess",\n        action="store_true",\n        default=False,\n        help="Apply stroke unsharp mask and background thresholding (default: False for Pure U-Net)",\n    )\n    args = parser.parse_args()\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    print(f"Device: {device}")\n\n    model = DocumentRestorationUNet().to(device)\n    if os.path.exists(args.model):\n        model.load_state_dict(torch.load(args.model, map_location=device, weights_only=True))\n        print(f"Loaded model from {args.model}")\n    else:\n        print(f"ERROR: Model not found at {args.model}.")\n        return\n\n    model.eval()\n\n    os.makedirs(args.output_dir, exist_ok=True)\n    os.makedirs(args.preview_dir, exist_ok=True)\n    os.makedirs(args.blind_out_dir, exist_ok=True)\n    comp_dir = os.path.join(args.output_dir, "comparison")\n    os.makedirs(comp_dir, exist_ok=True)\n\n    extensions = (".jpg", ".jpeg", ".png", ".webp", ".jfif")\n    image_paths = []\n    if os.path.exists(args.input_dir):\n        for entry in sorted(os.listdir(args.input_dir)):\n            full_p = os.path.join(args.input_dir, entry)\n            if os.path.isfile(full_p) and os.path.splitext(entry)[1].lower() in extensions:\n                image_paths.append(full_p)\n\n    print(f"Found {len(image_paths)} images in {args.input_dir}")\n\n    processed = 0\n    for idx, img_path in enumerate(image_paths, 1):\n        raw_bgr = load_image(img_path)\n        if raw_bgr is None:\n            print(f"  [{idx}/{len(image_paths)}] SKIP: Could not load image {img_path}")\n            continue\n\n        base_name = os.path.splitext(os.path.basename(img_path))[0]\n        is_blind_eval = base_name.lower() in ("real_11", "real_12")\n\n        # For blind evaluation images (real_11, real_12), strictly enforce 100% Pure U-Net\n        restored_bgr = restore_image_for_yolov8(\n            model,\n            raw_bgr,\n            device,\n            tile_size=args.tile_size,\n            overlap=args.overlap,\n            preserve_color=False if is_blind_eval else args.preserve_color,\n            refine_glyphs=False,\n            apply_postprocess=False if is_blind_eval else args.postprocess,\n        )\n\n        filename = f"{base_name}_unet_yolo.png"\n        save_image(os.path.join(args.output_dir, filename), restored_bgr)\n\n        target_path = os.path.join(args.target_dir, f"{base_name}.png")\n        has_valid_target = (not is_blind_eval) and os.path.exists(target_path)\n\n        if has_valid_target:\n            target_img = load_image(target_path)\n            preview_sheet = create_three_column_preview(raw_bgr, target_img, restored_bgr)\n        else:\n            preview_sheet = create_two_column_blind_preview(raw_bgr, restored_bgr)\n            save_image(os.path.join(args.blind_out_dir, f"{base_name}_pure_unet.png"), restored_bgr)\n            save_image(os.path.join(args.blind_out_dir, f"{base_name}_blind_comparison.png"), preview_sheet)\n\n        preview_path = os.path.join(args.preview_dir, f"{base_name}_unet_preview.png")\n        save_image(preview_path, preview_sheet)\n        save_image(os.path.join(comp_dir, f"comparison_{idx}_{base_name}.png"), preview_sheet)\n\n        processed += 1\n        mode_tag = "2-Col Blind Pure U-Net" if not has_valid_target else "3-Col Paired Preview"\n        print(f"  [{idx}/{len(image_paths)}] [{mode_tag}] Processed -> {preview_path}")\n\n    print(f"\\nFinished processing {processed} images.")\n    print(f"Restored YOLOv8 images: {args.output_dir}")\n    print(f"U-Net Previews: {args.preview_dir}")\n    print(f"Blind Test Pure U-Net Previews: {args.blind_out_dir}")\n\n\nif __name__ == "__main__":\n    main()\n\n'

GENERATE_CLEAN_TARGETS_CODE = 'import os\nimport re\nimport sys\nimport csv\nimport glob\nimport argparse\nimport logging\nfrom typing import Optional, List, Dict, Tuple\n\nimport cv2\nimport numpy as np\nimport torch\n\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))\nfrom training.restoration_dataset import (\n    load_image,\n    save_image,\n    unsharp_mask,\n    clean_target_normalization,\n    infer_single_image_tiled,\n)\nfrom training.train_restoration import DocumentRestorationUNet\n\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")\nlogger = logging.getLogger("generate_clean_targets")\n\nIMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp", ".jfif", ".bmp")\n\n\ndef generate_master_clean_target(\n    raw_gray: np.ndarray,\n    model: Optional[torch.nn.Module] = None,\n    device: Optional[torch.device] = None,\n    tile_size: int = 512,\n    overlap: int = 128,\n    use_nlm: bool = False,\n) -> np.ndarray:\n    """\n    Generate a high-quality Master Clean Target from a raw camera photo.\n\n    Supports Hybrid Mode (Signal Processing + Optional U-Net Prior):\n    - When `model` is provided, uses U-Net\'s tiled shadow-free prediction as a low-frequency\n      illumination prior to lift deep cast shadows (`ShadowCast`) before running the 5-step\n      Multi-Scale Morphological + Sauvola Soft Matting pipeline, preserving 100% of original\n      camera stroke sharpness and Vietnamese diacritics.\n    - When `model` is None, runs pure 5-step Signal Processing (`clean_target_normalization`).\n    """\n    if raw_gray.ndim == 3:\n        raw_gray = cv2.cvtColor(raw_gray, cv2.COLOR_BGR2GRAY)\n\n    if model is not None:\n        if device is None:\n            device = next(model.parameters()).device\n        unet_pred = infer_single_image_tiled(\n            model, raw_gray, device, tile_size=tile_size, overlap=overlap\n        )\n\n        # Estimate smooth low-frequency illumination envelopes from raw photo and U-Net prior\n        k_env = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (31, 31))\n        raw_env = cv2.GaussianBlur(\n            cv2.morphologyEx(raw_gray, cv2.MORPH_CLOSE, k_env).astype(np.float32),\n            (0, 0),\n            sigmaX=21,\n            sigmaY=21,\n        )\n        unet_env = cv2.GaussianBlur(\n            cv2.morphologyEx(unet_pred, cv2.MORPH_CLOSE, k_env).astype(np.float32),\n            (0, 0),\n            sigmaX=21,\n            sigmaY=21,\n        )\n\n        # Lift deep shadow regions using smooth U-Net illumination gain without blurring stroke edges\n        shadow_gain = np.clip(unet_env / np.maximum(raw_env, 15.0), 1.0, 3.2)\n        shadow_lifted = np.clip(raw_gray.astype(np.float32) * shadow_gain, 0.0, 255.0).astype(\n            np.uint8\n        )\n\n        clean_hybrid = clean_target_normalization(shadow_lifted, use_nlm=use_nlm)\n\n        # Suppress any faint shadow boundary transition where U-Net strongly predicts pure background\n        clean_hybrid = np.where(\n            (unet_pred > 238) & (clean_hybrid > 175), 255, clean_hybrid\n        ).astype(np.uint8)\n        return clean_hybrid\n\n    return clean_target_normalization(raw_gray, use_nlm=use_nlm)\n\n\ndef _find_best_zoom_patch(img_gray: np.ndarray, patch_ratio: float = 0.28) -> Tuple[int, int, int, int]:\n    """\n    Locate a text-rich region with high gradient density for the Zoom-in comparison patch.\n    Returns (x0, y0, pw, ph).\n    """\n    h, w = img_gray.shape[:2]\n    pw = max(32, min(w, int(w * patch_ratio)))\n    ph = max(32, min(h, int(h * patch_ratio)))\n    if w <= pw or h <= ph:\n        return 0, 0, w, h\n\n    gx = cv2.Sobel(img_gray, cv2.CV_32F, 1, 0, ksize=3)\n    gy = cv2.Sobel(img_gray, cv2.CV_32F, 0, 1, ksize=3)\n    grad_mag = np.hypot(gx, gy)\n\n    integral_grad = cv2.integral(grad_mag)\n    step_y = max(1, (h - ph) // 16)\n    step_x = max(1, (w - pw) // 16)\n\n    best_score = -1.0\n    best_xy = ((w - pw) // 2, (h - ph) // 2)\n\n    for y in range(0, h - ph + 1, step_y):\n        for x in range(0, w - pw + 1, step_x):\n            score = float(\n                integral_grad[y + ph, x + pw]\n                - integral_grad[y, x + pw]\n                - integral_grad[y + ph, x]\n                + integral_grad[y, x]\n            )\n            if score > best_score:\n                best_score = score\n                best_xy = (x, y)\n\n    return best_xy[0], best_xy[1], pw, ph\n\n\ndef create_preview_comparison(\n    raw_bgr: np.ndarray,\n    clean_gray: np.ndarray,\n    target_height: int = 560,\n) -> np.ndarray:\n    """\n    Create a 2-column visual comparison sheet:\n    [1. Raw Raspberry Pi Photo | 2. Generated Clean Target]\n    with a synchronized Zoom-in patch at the bottom to inspect Vietnamese diacritics and stroke edges.\n    """\n    if raw_bgr.ndim == 2:\n        raw_bgr = cv2.cvtColor(raw_bgr, cv2.COLOR_GRAY2BGR)\n    clean_bgr = (\n        cv2.cvtColor(clean_gray, cv2.COLOR_GRAY2BGR) if clean_gray.ndim == 2 else clean_gray.copy()\n    )\n\n    h, w = clean_gray.shape[:2]\n    if raw_bgr.shape[:2] != (h, w):\n        raw_bgr = cv2.resize(raw_bgr, (w, h), interpolation=cv2.INTER_AREA)\n\n    zx, zy, zw, zh = _find_best_zoom_patch(clean_gray)\n    raw_crop = raw_bgr[zy:zy + zh, zx:zx + zw].copy()\n    clean_crop = clean_bgr[zy:zy + zh, zx:zx + zw].copy()\n\n    raw_annotated = raw_bgr.copy()\n    clean_annotated = clean_bgr.copy()\n    box_thickness = max(2, min(h, w) // 250)\n    cv2.rectangle(raw_annotated, (zx, zy), (zx + zw, zy + zh), (0, 140, 255), box_thickness)\n    cv2.rectangle(clean_annotated, (zx, zy), (zx + zw, zy + zh), (0, 180, 0), box_thickness)\n\n    scale = target_height / float(max(h, 1))\n    disp_w = max(220, int(w * scale))\n\n    disp_raw = cv2.resize(raw_annotated, (disp_w, target_height), interpolation=cv2.INTER_AREA)\n    disp_clean = cv2.resize(clean_annotated, (disp_w, target_height), interpolation=cv2.INTER_AREA)\n\n    zoom_h = max(140, int(target_height * 0.36))\n    zoom_raw = cv2.resize(raw_crop, (disp_w, zoom_h), interpolation=cv2.INTER_CUBIC)\n    zoom_clean = cv2.resize(clean_crop, (disp_w, zoom_h), interpolation=cv2.INTER_CUBIC)\n\n    cv2.rectangle(zoom_raw, (0, 0), (disp_w - 1, zoom_h - 1), (0, 140, 255), 2)\n    cv2.rectangle(zoom_clean, (0, 0), (disp_w - 1, zoom_h - 1), (0, 180, 0), 2)\n\n    cv2.putText(\n        disp_raw,\n        "1. Raw Raspberry Pi Photo",\n        (12, 32),\n        cv2.FONT_HERSHEY_SIMPLEX,\n        0.75,\n        (0, 0, 255),\n        2,\n    )\n    cv2.putText(\n        disp_clean,\n        "2. Generated Clean Target",\n        (12, 32),\n        cv2.FONT_HERSHEY_SIMPLEX,\n        0.75,\n        (0, 150, 0),\n        2,\n    )\n    cv2.putText(\n        zoom_raw,\n        "Zoom-in: Raw Stroke & Diacritics",\n        (12, 26),\n        cv2.FONT_HERSHEY_SIMPLEX,\n        0.6,\n        (0, 140, 255),\n        2,\n    )\n    cv2.putText(\n        zoom_clean,\n        "Zoom-in: Clean Anti-Aliased Target",\n        (12, 26),\n        cv2.FONT_HERSHEY_SIMPLEX,\n        0.6,\n        (0, 160, 0),\n        2,\n    )\n\n    col_left = np.vstack([disp_raw, zoom_raw])\n    col_right = np.vstack([disp_clean, zoom_clean])\n    divider = np.full((col_left.shape[0], 4, 3), 200, dtype=np.uint8)\n    return np.hstack([col_left, divider, col_right])\n\n\ndef _get_next_original_id(original_dir: str) -> str:\n    """Find next sequential AnhXX identifier in datasets/original/ (e.g. Anh23)."""\n    os.makedirs(original_dir, exist_ok=True)\n    max_idx = 0\n    for fname in os.listdir(original_dir):\n        m = re.match(r"^Anh(\\d+)", fname, re.IGNORECASE)\n        if m:\n            max_idx = max(max_idx, int(m.group(1)))\n    next_idx = max(max_idx + 1, 1)\n    return f"Anh{next_idx:02d}"\n\n\ndef export_to_original_dir(\n    clean_gray: np.ndarray,\n    original_dir: str = "datasets/original",\n    source_id: Optional[str] = None,\n) -> Tuple[str, str]:\n    """\n    Export the approved Clean Target to `datasets/original/AnhXX.png`.\n    Returns (assigned_id, exported_path).\n    """\n    os.makedirs(original_dir, exist_ok=True)\n    assigned_id = source_id or _get_next_original_id(original_dir)\n    out_path = os.path.join(original_dir, f"{assigned_id}.png")\n    save_image(out_path, clean_gray)\n    return assigned_id, out_path\n\n\ndef register_pair_in_train_set(\n    raw_bgr: np.ndarray,\n    clean_gray: np.ndarray,\n    base_name: str,\n    dataset_root: str = "datasets",\n    source_id: Optional[str] = None,\n) -> Tuple[str, str]:\n    """\n    Automatically save the (raw_photo -> clean_target) pair into the training split:\n    - Saves raw photo to `datasets/processed/train/input/RealCamera_<base_name>.png`\n    - Saves clean target to `datasets/processed/train/target/<base_name>.png`\n    - Appends/updates rows in `datasets/metadata/train.csv` and `datasets/metadata/dataset_manifest.csv`\n      without duplicating existing entries.\n    """\n    safe_base = re.sub(r"[^\\w\\-]", "_", base_name).strip("_") or "real_doc"\n    src_id = source_id or safe_base\n\n    train_in_rel = f"processed/train/input/RealCamera_{safe_base}.png"\n    train_tgt_rel = f"processed/train/target/{safe_base}.png"\n\n    train_in_abs = os.path.join(dataset_root, train_in_rel)\n    train_tgt_abs = os.path.join(dataset_root, train_tgt_rel)\n\n    save_image(train_in_abs, raw_bgr)\n    save_image(train_tgt_abs, clean_gray)\n\n    meta_dir = os.path.join(dataset_root, "metadata")\n    os.makedirs(meta_dir, exist_ok=True)\n\n    # 1. Update train.csv\n    train_csv_path = os.path.join(meta_dir, "train.csv")\n    csv_in_val = f"datasets/{train_in_rel}".replace("\\\\", "/")\n    csv_tgt_val = f"datasets/{train_tgt_rel}".replace("\\\\", "/")\n    train_fieldnames = ["input", "target", "source_id", "degradation", "blur_level"]\n\n    existing_train_rows = []\n    if os.path.exists(train_csv_path):\n        with open(train_csv_path, "r", encoding="utf-8-sig", newline="") as f:\n            reader = csv.DictReader(f)\n            if reader.fieldnames:\n                train_fieldnames = list(reader.fieldnames)\n            for r in reader:\n                if r.get("input", "").strip().replace("\\\\", "/") != csv_in_val:\n                    existing_train_rows.append(r)\n\n    new_train_row = {\n        "input": csv_in_val,\n        "target": csv_tgt_val,\n        "source_id": src_id,\n        "degradation": "RealCamera",\n        "blur_level": "RealCamera_01",\n    }\n    existing_train_rows.append({k: new_train_row.get(k, "") for k in train_fieldnames})\n\n    with open(train_csv_path, "w", encoding="utf-8", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=train_fieldnames)\n        writer.writeheader()\n        writer.writerows(existing_train_rows)\n\n    # 2. Update dataset_manifest.csv\n    manifest_path = os.path.join(meta_dir, "dataset_manifest.csv")\n    manifest_fieldnames = [\n        "split",\n        "original_id",\n        "degradation_type",\n        "variant_id",\n        "input_path",\n        "target_path",\n    ]\n    existing_manifest_rows = []\n    if os.path.exists(manifest_path):\n        with open(manifest_path, "r", encoding="utf-8-sig", newline="") as f:\n            reader = csv.DictReader(f)\n            if reader.fieldnames:\n                manifest_fieldnames = list(reader.fieldnames)\n            for r in reader:\n                if r.get("input_path", "").strip().replace("\\\\", "/") != train_in_rel:\n                    existing_manifest_rows.append(r)\n\n    new_manifest_row = {\n        "split": "train",\n        "original_id": src_id,\n        "degradation_type": "RealCamera",\n        "variant_id": "01",\n        "input_path": train_in_rel,\n        "target_path": train_tgt_rel,\n    }\n    existing_manifest_rows.append({k: new_manifest_row.get(k, "") for k in manifest_fieldnames})\n\n    with open(manifest_path, "w", encoding="utf-8", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=manifest_fieldnames)\n        writer.writeheader()\n        writer.writerows(existing_manifest_rows)\n\n    return train_in_abs, train_tgt_abs\n\n\ndef process_real_directory(\n    input_dir: str = "datasets/real",\n    model_path: str = "models/restoration_best.pth",\n    use_unet: bool = True,\n    use_nlm: bool = False,\n    export_to_original: bool = False,\n    save_to_train: bool = True,\n    dataset_root: str = "datasets",\n    tile_size: int = 512,\n    overlap: int = 128,\n) -> List[Dict[str, str]]:\n    """\n    Scan all raw camera photos in `input_dir`, generate Clean Targets + Preview Comparisons,\n    automatically register (raw_photo -> clean_target) pairs into the training set, and\n    optionally export sequential AnhXX.png files to `datasets/original/`.\n    """\n    os.makedirs(input_dir, exist_ok=True)\n    clean_targets_dir = os.path.join(input_dir, "clean_targets")\n    preview_dir = os.path.join(input_dir, "preview_comparisons")\n    os.makedirs(clean_targets_dir, exist_ok=True)\n    os.makedirs(preview_dir, exist_ok=True)\n\n    # Collect raw image files directly inside input_dir (do not recurse into output subfolders)\n    raw_files = []\n    for entry in sorted(os.listdir(input_dir)):\n        full_p = os.path.join(input_dir, entry)\n        if os.path.isfile(full_p) and os.path.splitext(entry)[1].lower() in IMAGE_EXTENSIONS:\n            raw_files.append(full_p)\n\n    if not raw_files:\n        logger.warning(f"No raw images found in \'{input_dir}\'.")\n        return []\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = None\n    if use_unet and model_path and os.path.exists(model_path):\n        try:\n            model = DocumentRestorationUNet().to(device)\n            model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))\n            model.eval()\n            logger.info(f"Loaded U-Net prior from \'{model_path}\' on {device} (Hybrid Mode enabled).")\n        except Exception as e:\n            logger.warning(f"Could not load U-Net model from \'{model_path}\' ({e}). Using pure Signal Processing mode.")\n            model = None\n    else:\n        logger.info("Running in Signal Processing Master Ground-Truth mode (5-step Sauvola Soft Matting).")\n\n    results: List[Dict[str, str]] = []\n    original_dir = os.path.join(dataset_root, "original")\n\n    for idx, img_path in enumerate(raw_files, 1):\n        raw_bgr = load_image(img_path)\n        if raw_bgr is None:\n            logger.warning(f"[{idx}/{len(raw_files)}] Failed to load image: {img_path}")\n            continue\n\n        base_name = os.path.splitext(os.path.basename(img_path))[0]\n        raw_gray = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)\n\n        clean_target = generate_master_clean_target(\n            raw_gray,\n            model=model,\n            device=device,\n            tile_size=tile_size,\n            overlap=overlap,\n            use_nlm=use_nlm,\n        )\n\n        # 1. Save Clean Target to datasets/real/clean_targets/<base_name>.png\n        clean_out_path = os.path.join(clean_targets_dir, f"{base_name}.png")\n        save_image(clean_out_path, clean_target)\n\n        # 2. Save 2-column comparison + Zoom-in patch to datasets/real/preview_comparisons/\n        preview_sheet = create_preview_comparison(raw_bgr, clean_target)\n        preview_out_path = os.path.join(preview_dir, f"{base_name}_preview.png")\n        save_image(preview_out_path, preview_sheet)\n\n        assigned_id = base_name\n        exported_orig_path = ""\n        if export_to_original:\n            assigned_id, exported_orig_path = export_to_original_dir(\n                clean_target, original_dir=original_dir\n            )\n\n        train_in_path = ""\n        train_tgt_path = ""\n        if save_to_train:\n            train_in_path, train_tgt_path = register_pair_in_train_set(\n                raw_bgr,\n                clean_target,\n                base_name=base_name,\n                dataset_root=dataset_root,\n                source_id=assigned_id,\n            )\n\n        rec = {\n            "raw_path": img_path,\n            "clean_target_path": clean_out_path,\n            "preview_path": preview_out_path,\n            "train_input_path": train_in_path,\n            "train_target_path": train_tgt_path,\n            "exported_original_path": exported_orig_path,\n            "source_id": assigned_id,\n        }\n        results.append(rec)\n        logger.info(\n            f"[{idx}/{len(raw_files)}] Generated Clean Target: {clean_out_path} | Preview: {preview_out_path}"\n            + (f" | Exported: {exported_orig_path}" if exported_orig_path else "")\n            + (f" | Added to train.csv ({assigned_id})" if train_in_path else "")\n        )\n\n    return results\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Master Ground-Truth Clean Target Generator for Real Camera Photos (datasets/real)"\n    )\n    parser.add_argument(\n        "--input-dir",\n        type=str,\n        default="datasets/real",\n        help="Directory containing raw Raspberry Pi camera photos",\n    )\n    parser.add_argument(\n        "--dataset-root",\n        type=str,\n        default="datasets",\n        help="Root datasets directory containing processed/ and metadata/",\n    )\n    parser.add_argument(\n        "--model",\n        type=str,\n        default="models/restoration_best.pth",\n        help="Optional U-Net checkpoint for Hybrid Shadow Removal prior",\n    )\n    parser.add_argument(\n        "--no-unet",\n        action="store_true",\n        help="Disable U-Net prior and use pure 5-step Signal Processing pipeline",\n    )\n    parser.add_argument(\n        "--use-nlm",\n        action="store_true",\n        help="Use cv2.fastNlMeansDenoising instead of bilateralFilter in Step 1",\n    )\n    parser.add_argument(\n        "--export-to-original",\n        action="store_true",\n        help="Automatically number next AnhXX.png and copy clean target into datasets/original/",\n    )\n    parser.add_argument(\n        "--no-save-to-train",\n        action="store_false",\n        dest="save_to_train",\n        default=True,\n        help="Skip automatic saving of (raw_photo -> clean_target) into train split",\n    )\n    parser.add_argument("--tile-size", type=int, default=512, help="Tile size for U-Net prior")\n    parser.add_argument("--overlap", type=int, default=128, help="Tile overlap for U-Net prior")\n    args = parser.parse_args()\n\n    process_real_directory(\n        input_dir=args.input_dir,\n        model_path=args.model,\n        use_unet=not args.no_unet,\n        use_nlm=args.use_nlm,\n        export_to_original=args.export_to_original,\n        save_to_train=args.save_to_train,\n        dataset_root=args.dataset_root,\n        tile_size=args.tile_size,\n        overlap=args.overlap,\n    )\n\n\nif __name__ == "__main__":\n    main()\n'

PREPARE_REAL_PAIRS_CODE = 'import os\nimport sys\nimport csv\nimport glob\nimport shutil\nimport argparse\nimport logging\nfrom typing import List, Dict, Tuple\n\nimport cv2\nimport numpy as np\n\nsys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))\nfrom training.restoration_dataset import (\n    load_image,\n    save_image,\n    unsharp_mask,\n    _sauvola_threshold_integral,\n)\n\nlogging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")\nlogger = logging.getLogger("prepare_real_pairs")\n\n\ndef refine_real_clean_target(raw_bgr: np.ndarray) -> np.ndarray:\n    """\n    Create a Master Clean Target (`clean_real_target`) from a real camera document photo:\n    1. Multi-Scale Morphological Closing (`21x21`, `45x45`, `75x75`) to flatten uneven background\n       illumination and book spine shadows to pure white `255`.\n    2. Bleed-through & Watermark Suppression: Regions with normalized intensity `> 210` and low\n       edge gradient are pushed cleanly to `255`, while 100% protecting front-side printed text\n       and thin handwritten ballpoint pen strokes.\n    3. Stroke Contrast & Sharpness Enhancement: Maps foreground stroke cores to deep crisp ink\n       (`0 -> 30`) with smooth anti-aliased edges.\n    """\n    if raw_bgr.ndim == 2:\n        raw_gray = raw_bgr.copy()\n        blue_ink_boost = raw_gray\n    else:\n        b_ch, g_ch, r_ch = cv2.split(raw_bgr)\n        raw_gray = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)\n        # Protect blue ballpoint pen strokes and blue links (where B > R and G is darker than B)\n        blue_excess = b_ch.astype(np.float32) - np.maximum(r_ch.astype(np.float32), g_ch.astype(np.float32))\n        rg_mean = 0.5 * (r_ch.astype(np.float32) + g_ch.astype(np.float32))\n        blue_ink_boost = np.where(\n            blue_excess > 12.0,\n            np.minimum(raw_gray.astype(np.float32), rg_mean),\n            raw_gray.astype(np.float32),\n        ).astype(np.uint8)\n\n    # Step 1: Edge-Preserving Pre-Denoising\n    denoised = cv2.bilateralFilter(blue_ink_boost, d=5, sigmaColor=20, sigmaSpace=20)\n\n    # Step 2: Multi-Scale Morphological Closing for Background Illumination Flattening\n    k_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (21, 21))\n    k_med = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (45, 45))\n    k_large = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (75, 75))\n\n    bg_small = cv2.morphologyEx(denoised, cv2.MORPH_CLOSE, k_small)\n    bg_med = cv2.morphologyEx(denoised, cv2.MORPH_CLOSE, k_med)\n    bg_large = cv2.morphologyEx(denoised, cv2.MORPH_CLOSE, k_large)\n    bg = np.maximum(np.maximum(bg_small, bg_med), bg_large)\n    bg = cv2.GaussianBlur(bg, (0, 0), sigmaX=15, sigmaY=15)\n\n    norm = np.clip(\n        (denoised.astype(np.float32) / np.maximum(bg.astype(np.float32), 15.0)) * 255.0,\n        0.0,\n        255.0,\n    )\n\n    #Also compute standard luminance normalized reflectance to detect pastel/orange watermarks\n    if raw_bgr.ndim == 3:\n        denoised_luma = cv2.bilateralFilter(raw_gray, d=5, sigmaColor=20, sigmaSpace=20)\n        bg_luma = cv2.GaussianBlur(\n            np.maximum(\n                cv2.morphologyEx(denoised_luma, cv2.MORPH_CLOSE, k_small),\n                cv2.morphologyEx(denoised_luma, cv2.MORPH_CLOSE, k_med),\n            ),\n            (0, 0),\n            sigmaX=15,\n            sigmaY=15,\n        )\n        norm_luma = np.clip(\n            (denoised_luma.astype(np.float32) / np.maximum(bg_luma.astype(np.float32), 15.0)) * 255.0,\n            0.0,\n            255.0,\n        )\n    else:\n        norm_luma = norm\n\n    # Step 3: Edge Gradient & Multi-Scale Sauvola Thresholding\n    gx = cv2.Sobel(norm, cv2.CV_32F, 1, 0, ksize=3)\n    gy = cv2.Sobel(norm, cv2.CV_32F, 0, 1, ksize=3)\n    grad_mag = np.hypot(gx, gy)\n    k_grad = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))\n    local_grad = cv2.dilate(grad_mag, k_grad)\n\n    t_small = _sauvola_threshold_integral(norm, window_size=15, k=0.14, r=128.0)\n    t_med = _sauvola_threshold_integral(norm, window_size=31, k=0.18, r=128.0)\n    t_sauvola = np.maximum(t_small, t_med)\n\n    # Bleed-through and watermark suppression:\n    # - Regions with normalized intensity > 210 and low edge gradient (< 180) are pushed to 255\n    # - Regions with norm > 220 or above Sauvola threshold are background (255)\n    # - Thin ballpoint pen strokes have sharp local gradients (>= 180) or darker cores (norm <= 210)\n    is_bleed_or_watermark = (\n        (norm_luma > 210.0) & (local_grad < 180.0)\n    ) | (norm > 220.0) | ((norm - t_sauvola) > 4.0)\n\n    fg_mask = ~is_bleed_or_watermark\n\n    # Connected-component filtering to remove isolated faint bleed-through / dust islands\n    fg_u8 = fg_mask.astype(np.uint8)\n    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(fg_u8, connectivity=8)\n    lap_g = (\n        np.abs(cv2.Laplacian(raw_bgr[:, :, 1].astype(np.float32), cv2.CV_32F, ksize=3))\n        if raw_bgr.ndim == 3\n        else np.abs(cv2.Laplacian(raw_gray.astype(np.float32), cv2.CV_32F, ksize=3))\n    )\n    for lbl in range(1, num_labels):\n        area = stats[lbl, cv2.CC_STAT_AREA]\n        x0 = stats[lbl, cv2.CC_STAT_LEFT]\n        y0 = stats[lbl, cv2.CC_STAT_TOP]\n        w0 = stats[lbl, cv2.CC_STAT_WIDTH]\n        h0 = stats[lbl, cv2.CC_STAT_HEIGHT]\n        roi_mask = labels[y0:y0 + h0, x0:x0 + w0] == lbl\n        roi_norm = norm_luma[y0:y0 + h0, x0:x0 + w0][roi_mask]\n        roi_grad = grad_mag[y0:y0 + h0, x0:x0 + w0][roi_mask]\n\n        min_val = float(np.min(roi_norm))\n        max_g = float(np.max(roi_grad))\n        # Remove micro-speckles or faint diffuse bleed-through blobs\n        if (area <= 3 and min_val > 145.0) or (min_val > 206.0 and max_g < 210.0):\n            fg_mask[y0:y0 + h0, x0:x0 + w0][roi_mask] = False\n            continue\n\n        # Suppress diffuse back-side red stamp bleed-through (high Red channel, low Laplacian sharpness)\n        if raw_bgr.ndim == 3:\n            roi_r = raw_bgr[y0:y0 + h0, x0:x0 + w0, 2][roi_mask].astype(np.float32)\n            roi_g = raw_bgr[y0:y0 + h0, x0:x0 + w0, 1][roi_mask].astype(np.float32)\n            roi_b = raw_bgr[y0:y0 + h0, x0:x0 + w0, 0][roi_mask].astype(np.float32)\n            red_exc = float(np.mean(roi_r - 0.5 * (roi_g + roi_b)))\n            if red_exc > 25.0 and float(np.min(roi_r)) > 88.0:\n                roi_lap = float(np.percentile(lap_g[y0:y0 + h0, x0:x0 + w0][roi_mask], 95))\n                if roi_lap < 185.0 and float(np.percentile(roi_g, 10)) > 28.0:\n                    fg_mask[y0:y0 + h0, x0:x0 + w0][roi_mask] = False\n\n    # Step 4: Smooth Anti-Aliased Contrast Mapping (Stroke Cores -> [0, 30])\n    matted_f = np.full_like(norm, 255.0, dtype=np.float32)\n    if np.any(fg_mask):\n        fg_vals = norm[fg_mask]\n        black_floor = float(np.clip(np.percentile(fg_vals, 25), 35.0, 155.0))\n        upper_bound = np.minimum(t_sauvola + 4.0, 218.0)\n        span = np.maximum(upper_bound - black_floor, 18.0)\n\n        alpha = np.clip((upper_bound - norm) / span, 0.0, 1.0)\n        alpha_boosted = np.power(alpha, 0.62)\n        # Map stroke cores cleanly into [0, 30] and anti-aliased edges smoothly to 255\n        stroke_val = 255.0 * (1.0 - alpha_boosted) * 0.88\n        matted_f = np.where(fg_mask, stroke_val, 255.0)\n\n    matted = np.clip(matted_f, 0.0, 255.0).astype(np.uint8)\n\n    # Step 5: Foreground Unsharp Masking for crisp stroke edges\n    sharpened = unsharp_mask(matted, amount=0.75, radius=1.0)\n    clean_target = np.where(matted < 245, sharpened, 255).astype(np.uint8)\n    return clean_target\n\n\nBLIND_EVAL_BASENAMES = {"real_11", "real_12"}\n\n\ndef _is_blind_eval_name(name_or_path: str) -> bool:\n    stem = os.path.splitext(os.path.basename(name_or_path))[0].lower()\n    return stem in BLIND_EVAL_BASENAMES\n\n\ndef purge_blind_eval_from_real_pairs(dataset_root: str = "datasets") -> List[str]:\n    """\n    Strictly remove `real_11.*` and `real_12.*` from `datasets/processed/real_pairs/{input,target}/`\n    if present, preventing any data leakage into the training set.\n    """\n    removed = []\n    for sub in ("input", "target"):\n        folder = os.path.join(dataset_root, "processed", "real_pairs", sub)\n        if not os.path.exists(folder):\n            continue\n        for fname in os.listdir(folder):\n            if _is_blind_eval_name(fname):\n                fpath = os.path.join(folder, fname)\n                try:\n                    os.remove(fpath)\n                    removed.append(fpath)\n                    logger.warning(f"Removed leaked blind-eval file from real_pairs: {fpath}")\n                except OSError:\n                    pass\n    return removed\n\n\ndef update_metadata_with_real_pairs(\n    real_records: List[Dict[str, str]],\n    dataset_root: str = "datasets",\n    train_ratio: float = 1.0,\n    train_oversample: int = 3,\n) -> Tuple[int, int]:\n    """\n    Put valid `real_pairs` into `datasets/metadata/train.csv` (with `oversample x3`),\n    strictly excluding `real_11` and `real_12`.\n    Also registers `real_12.jpg` in `val.csv` and `real_11.jpg` in `test.csv` as\n    unpaired blind evaluation entries (`target=""`, `degradation="unpaired_real"`).\n    Idempotent: removes previous `real_pairs` and `unpaired_real` entries before appending.\n    """\n    meta_dir = os.path.join(dataset_root, "metadata")\n    os.makedirs(meta_dir, exist_ok=True)\n\n    # Filter out any accidental blind eval records\n    safe_records = [\n        r for r in real_records\n        if not _is_blind_eval_name(r.get("source_id", ""))\n        and not _is_blind_eval_name(r.get("csv_input", ""))\n    ]\n\n    n_total = len(safe_records)\n    if train_ratio >= 1.0:\n        n_train = n_total\n    else:\n        n_train = max(1, int(round(n_total * train_ratio))) if n_total > 1 else n_total\n    train_records = safe_records[:n_train]\n    val_records = safe_records[n_train:]\n\n    def _is_real_pair_or_blind_row(row: Dict[str, str]) -> bool:\n        row_str = " ".join(str(v) for v in row.values()).lower().replace("\\\\", "/")\n        deg = (row.get("degradation") or row.get("degradation_type") or "").lower()\n        return (\n            "real_pairs" in row_str\n            or deg in ("real_pairs", "unpaired_real")\n            or "real_11" in row_str\n            or "real_12" in row_str\n        )\n\n    # 1. Update train.csv (strictly no real_11 or real_12)\n    train_csv_path = os.path.join(meta_dir, "train.csv")\n    train_fields = ["input", "target", "source_id", "degradation", "blur_level"]\n    existing_train = []\n    if os.path.exists(train_csv_path):\n        with open(train_csv_path, "r", encoding="utf-8-sig", newline="") as f:\n            reader = csv.DictReader(f)\n            if reader.fieldnames:\n                train_fields = list(reader.fieldnames)\n            existing_train = [r for r in reader if not _is_real_pair_or_blind_row(r)]\n\n    added_train = 0\n    for rec in train_records:\n        for rep in range(1, train_oversample + 1):\n            row_data = {\n                "input": rec["csv_input"],\n                "target": rec["csv_target"],\n                "source_id": rec["source_id"],\n                "degradation": "real_pairs",\n                "blur_level": f"real_pairs_rep{rep:02d}",\n            }\n            existing_train.append({k: row_data.get(k, "") for k in train_fields})\n            added_train += 1\n\n    with open(train_csv_path, "w", encoding="utf-8", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=train_fields)\n        writer.writeheader()\n        writer.writerows(existing_train)\n\n    # 2. Update val.csv (include unpaired real_12.jpg if present in datasets/processed/val/input/)\n    val_csv_path = os.path.join(meta_dir, "val.csv")\n    val_fields = ["input", "target", "source_id", "degradation", "blur_level"]\n    existing_val = []\n    if os.path.exists(val_csv_path):\n        with open(val_csv_path, "r", encoding="utf-8-sig", newline="") as f:\n            reader = csv.DictReader(f)\n            if reader.fieldnames:\n                val_fields = list(reader.fieldnames)\n            existing_val = [r for r in reader if not _is_real_pair_or_blind_row(r)]\n\n    added_val = 0\n    for rec in val_records:\n        row_data = {\n            "input": rec["csv_input"],\n            "target": rec["csv_target"],\n            "source_id": rec["source_id"],\n            "degradation": "real_pairs",\n            "blur_level": "real_pairs_val",\n        }\n        existing_val.append({k: row_data.get(k, "") for k in val_fields})\n        added_val += 1\n\n    val_real12_abs = os.path.join(dataset_root, "processed", "val", "input", "real_12.jpg")\n    if os.path.exists(val_real12_abs):\n        row_real12 = {\n            "input": "datasets/processed/val/input/real_12.jpg",\n            "target": "",\n            "source_id": "real_12",\n            "degradation": "unpaired_real",\n            "blur_level": "real_blind_val",\n        }\n        existing_val.append({k: row_real12.get(k, "") for k in val_fields})\n\n    with open(val_csv_path, "w", encoding="utf-8", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=val_fields)\n        writer.writeheader()\n        writer.writerows(existing_val)\n\n    # 3. Update test.csv (include unpaired real_11.jpg if present in datasets/processed/test/input/)\n    test_csv_path = os.path.join(meta_dir, "test.csv")\n    test_fields = ["input", "target", "source_id", "degradation", "blur_level"]\n    existing_test = []\n    if os.path.exists(test_csv_path):\n        with open(test_csv_path, "r", encoding="utf-8-sig", newline="") as f:\n            reader = csv.DictReader(f)\n            if reader.fieldnames:\n                test_fields = list(reader.fieldnames)\n            existing_test = [r for r in reader if not _is_real_pair_or_blind_row(r)]\n\n        test_real11_abs = os.path.join(dataset_root, "processed", "test", "input", "real_11.jpg")\n        if os.path.exists(test_real11_abs):\n            row_real11 = {\n                "input": "datasets/processed/test/input/real_11.jpg",\n                "target": "",\n                "source_id": "real_11",\n                "degradation": "unpaired_real",\n                "blur_level": "real_blind_test",\n            }\n            existing_test.append({k: row_real11.get(k, "") for k in test_fields})\n\n        with open(test_csv_path, "w", encoding="utf-8", newline="") as f:\n            writer = csv.DictWriter(f, fieldnames=test_fields)\n            writer.writeheader()\n            writer.writerows(existing_test)\n\n    # 4. Sync dataset_manifest.csv\n    manifest_path = os.path.join(meta_dir, "dataset_manifest.csv")\n    manifest_fields = [\n        "split",\n        "original_id",\n        "degradation_type",\n        "variant_id",\n        "input_path",\n        "target_path",\n    ]\n    existing_manifest = []\n    if os.path.exists(manifest_path):\n        with open(manifest_path, "r", encoding="utf-8-sig", newline="") as f:\n            reader = csv.DictReader(f)\n            if reader.fieldnames:\n                manifest_fields = list(reader.fieldnames)\n            existing_manifest = [r for r in reader if not _is_real_pair_or_blind_row(r)]\n\n    for rec in train_records:\n        for rep in range(1, train_oversample + 1):\n            m_row = {\n                "split": "train",\n                "original_id": rec["source_id"],\n                "degradation_type": "real_pairs",\n                "variant_id": f"{rep:02d}",\n                "input_path": rec["rel_input"],\n                "target_path": rec["rel_target"],\n            }\n            existing_manifest.append({k: m_row.get(k, "") for k in manifest_fields})\n\n    for rec in val_records:\n        m_row = {\n            "split": "val",\n            "original_id": rec["source_id"],\n            "degradation_type": "real_pairs",\n            "variant_id": "01",\n            "input_path": rec["rel_input"],\n            "target_path": rec["rel_target"],\n        }\n        existing_manifest.append({k: m_row.get(k, "") for k in manifest_fields})\n\n    if os.path.exists(os.path.join(dataset_root, "processed", "val", "input", "real_12.jpg")):\n        m_val_blind = {\n            "split": "val",\n            "original_id": "real_12",\n            "degradation_type": "unpaired_real",\n            "variant_id": "01",\n            "input_path": "processed/val/input/real_12.jpg",\n            "target_path": "",\n        }\n        existing_manifest.append({k: m_val_blind.get(k, "") for k in manifest_fields})\n\n    if os.path.exists(os.path.join(dataset_root, "processed", "test", "input", "real_11.jpg")):\n        m_test_blind = {\n            "split": "test",\n            "original_id": "real_11",\n            "degradation_type": "unpaired_real",\n            "variant_id": "01",\n            "input_path": "processed/test/input/real_11.jpg",\n            "target_path": "",\n        }\n        existing_manifest.append({k: m_test_blind.get(k, "") for k in manifest_fields})\n\n    with open(manifest_path, "w", encoding="utf-8", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=manifest_fields)\n        writer.writeheader()\n        writer.writerows(existing_manifest)\n\n    return added_train, added_val\n\n\ndef prepare_real_pairs(\n    real_dir: str = "datasets/real",\n    dataset_root: str = "datasets",\n    train_ratio: float = 1.0,\n    train_oversample: int = 3,\n) -> List[Dict[str, str]]:\n    """\n    Prepare (raw_real_input -> clean_real_target) pairs for training:\n    - Strictly excludes `real_11.jpg` and `real_12.jpg` from `real_pairs` and `train.csv`\n      (and removes them from `datasets/processed/real_pairs/` if accidentally present).\n    - Scans `datasets/processed/real_pairs/input/` (and `datasets/real/`) for the 18 training\n      `real_01..10`, `real_13..20` images + 10 `IMG20261022*.jpg` images.\n    - Generates clean targets via `refine_real_clean_target` into `datasets/processed/real_pairs/target/`.\n    - Adds all valid `real_pairs` to `datasets/metadata/train.csv` with `oversample x3`.\n    """\n    input_out_dir = os.path.join(dataset_root, "processed", "real_pairs", "input")\n    target_out_dir = os.path.join(dataset_root, "processed", "real_pairs", "target")\n    os.makedirs(input_out_dir, exist_ok=True)\n    os.makedirs(target_out_dir, exist_ok=True)\n\n    # 1. Prevent data leakage: remove real_11 / real_12 if accidentally placed in real_pairs\n    purge_blind_eval_from_real_pairs(dataset_root=dataset_root)\n\n    # 2. Sync valid training images from real_dir (IMG*.jpg and real_*.jpg except real_11 & real_12)\n    #    into datasets/processed/real_pairs/input/ if not already there\n    if os.path.exists(real_dir):\n        for pat in ("IMG*.jpg", "real_*.jpg"):\n            for src_p in sorted(glob.glob(os.path.join(real_dir, pat))):\n                if _is_blind_eval_name(src_p):\n                    continue\n                dst_p = os.path.join(input_out_dir, os.path.basename(src_p))\n                if not os.path.exists(dst_p):\n                    shutil.copy2(src_p, dst_p)\n\n    # 3. Collect all valid training inputs from datasets/processed/real_pairs/input/\n    candidate_paths = []\n    for entry in sorted(os.listdir(input_out_dir)):\n        full_p = os.path.join(input_out_dir, entry)\n        if not os.path.isfile(full_p):\n            continue\n        if _is_blind_eval_name(entry):\n            continue\n        ext = os.path.splitext(entry)[1].lower()\n        if ext in (".jpg", ".jpeg", ".png", ".webp", ".jfif"):\n            candidate_paths.append(full_p)\n\n    if not candidate_paths:\n        logger.warning(f"No valid real training images found in \'{input_out_dir}\'.")\n        return []\n\n    logger.info(\n        f"Found {len(candidate_paths)} valid real training images in \'{input_out_dir}\' "\n        f"(excluded blind-eval images: {sorted(BLIND_EVAL_BASENAMES)})."\n    )\n    records: List[Dict[str, str]] = []\n\n    for idx, img_path in enumerate(candidate_paths, 1):\n        raw_bgr = load_image(img_path)\n        if raw_bgr is None:\n            logger.warning(f"[{idx}/{len(candidate_paths)}] Could not load {img_path}")\n            continue\n\n        base_name = os.path.splitext(os.path.basename(img_path))[0]\n        ext = os.path.splitext(img_path)[1].lower() or ".jpg"\n\n        rel_input = f"processed/real_pairs/input/{base_name}{ext}"\n        rel_target = f"processed/real_pairs/target/{base_name}.png"\n        abs_input = os.path.join(dataset_root, rel_input)\n        abs_target = os.path.join(dataset_root, rel_target)\n\n        # Generate refined clean target using refine_real_clean_target\n        clean_target = refine_real_clean_target(raw_bgr)\n        save_image(abs_target, clean_target)\n\n        records.append({\n            "source_id": base_name,\n            "raw_path": img_path,\n            "input_path": abs_input,\n            "target_path": abs_target,\n            "rel_input": rel_input.replace("\\\\", "/"),\n            "rel_target": rel_target.replace("\\\\", "/"),\n            "csv_input": f"datasets/{rel_input}".replace("\\\\", "/"),\n            "csv_target": f"datasets/{rel_target}".replace("\\\\", "/"),\n        })\n        logger.info(\n            f"[{idx}/{len(candidate_paths)}] Prepared pair: {rel_input} -> {rel_target} "\n            f"(shape={clean_target.shape[1]}x{clean_target.shape[0]})"\n        )\n\n    added_train, added_val = update_metadata_with_real_pairs(\n        records,\n        dataset_root=dataset_root,\n        train_ratio=train_ratio,\n        train_oversample=train_oversample,\n    )\n    logger.info(\n        f"Metadata updated: +{added_train} rows in train.csv ({int(added_train // max(train_oversample, 1))} unique x{train_oversample} oversample), "\n        f"+{added_val} paired rows in val.csv (plus unpaired real_12 in val.csv & real_11 in test.csv)."\n    )\n    return records\n\n\ndef main():\n    parser = argparse.ArgumentParser(\n        description="Prepare (raw_real_input -> clean_real_target) pairs for U-Net training (excluding real_11 & real_12)"\n    )\n    parser.add_argument("--real-dir", type=str, default="datasets/real", help="Directory with raw real camera photos")\n    parser.add_argument("--dataset-root", type=str, default="datasets", help="Root datasets directory")\n    parser.add_argument("--train-ratio", type=float, default=1.0, help="Fraction of real_pairs for train split (default 1.0 = all 28 pairs in train.csv)")\n    parser.add_argument("--oversample", type=int, default=3, help="Oversampling factor for real pairs in train.csv")\n    args = parser.parse_args()\n\n    prepare_real_pairs(\n        real_dir=args.real_dir,\n        dataset_root=args.dataset_root,\n        train_ratio=args.train_ratio,\n        train_oversample=args.oversample,\n    )\n\n\nif __name__ == "__main__":\n    main()\n\n'

with open(f'{PROJECT_DIR}/training/restoration_dataset.py', 'w', encoding='utf-8') as f:
    f.write(RESTORATION_DATASET_CODE)
with open(f'{PROJECT_DIR}/training/train_restoration.py', 'w', encoding='utf-8') as f:
    f.write(TRAIN_RESTORATION_CODE)
with open(f'{PROJECT_DIR}/training/evaluate_restoration.py', 'w', encoding='utf-8') as f:
    f.write(EVALUATE_RESTORATION_CODE)
with open(f'{PROJECT_DIR}/training/infer_restoration.py', 'w', encoding='utf-8') as f:
    f.write(INFER_RESTORATION_CODE)
with open(f'{PROJECT_DIR}/training/generate_clean_targets.py', 'w', encoding='utf-8') as f:
    f.write(GENERATE_CLEAN_TARGETS_CODE)
with open(f'{PROJECT_DIR}/training/prepare_real_pairs.py', 'w', encoding='utf-8') as f:
    f.write(PREPARE_REAL_PAIRS_CODE)

print('Đã cập nhật thành công 6 file code mới nhất vào:', f'{PROJECT_DIR}/training/')

for mod, pkg in [('cv2', 'opencv-python-headless'), ('PIL', 'pillow'), ('pandas', 'pandas')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        get_ipython().system(f'pip -q install {pkg}')

import random, json, math, time, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torch.nn as nn, torch.optim as optim
import training.restoration_dataset as rd_mod
import training.train_restoration as tr_mod
import training.evaluate_restoration as ev_mod
import training.infer_restoration as inf_mod
import training.generate_clean_targets as gct_mod
import training.prepare_real_pairs as prp_mod
importlib.reload(rd_mod)
importlib.reload(tr_mod)
importlib.reload(ev_mod)
importlib.reload(inf_mod)
importlib.reload(gct_mod)
importlib.reload(prp_mod)

from training.restoration_dataset import (
    DocumentRestorationDataset,
    load_image,
    save_image,
    resize_with_padding,
    remove_padding,
    unsharp_mask,
    infer_single_image_tiled,
    apply_hard_augmentation,
    clean_target_normalization,
    GEOMETRIC_DISTORTION_KEYWORDS,
)
from training.train_restoration import (
    DocumentRestorationUNet,
    DocumentCleanLoss,
    BleedThroughSuppressionLoss,
    BimodalStrokeLoss,
    SobelLoss,
    SSIMLoss,
    calculate_psnr,
)
from training.infer_restoration import (
    restore_image_for_yolov8,
    create_three_column_preview,
    create_two_column_blind_preview,
)
from training.prepare_real_pairs import refine_real_clean_target, prepare_real_pairs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Thiết bị thực thi:', DEVICE)


In [ ]:
#@title 4. Kiểm tra dữ liệu đã gộp: train.csv (267 mẫu = 183 gốc + 28 real_pairs x3), val.csv (24 mẫu có target + real_12.jpg blind), test.csv (16 mẫu có target + real_11.jpg blind)
if not os.path.exists(f'{DATASET_ROOT}/processed/real_pairs/target/real_20.png'):
    print('Đang tạo cặp dữ liệu thực tế real_pairs (loại trừ real_11 & real_12) ...')
    prepare_real_pairs()

df_manifest = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
df_train    = pd.read_csv(TRAIN_CSV, dtype=str, keep_default_na=False)
df_val      = pd.read_csv(VAL_CSV, dtype=str, keep_default_na=False)
df_test     = pd.read_csv(TEST_CSV, dtype=str, keep_default_na=False)

print(f'Tổng số dòng trong dataset_manifest.csv: {len(df_manifest)}')
print(f'  - train.csv : {len(df_train)} mẫu (183 mẫu gốc + 84 mẫu từ 28 cặp real_pairs oversample x3)')
print(f'  - val.csv   : {len(df_val)} dòng (24 mẫu có target + 1 ảnh blind val real_12.jpg)')
print(f'  - test.csv  : {len(df_test)} dòng (16 mẫu có target + 1 ảnh blind test real_11.jpg)')

# Kiểm tra chống rò rỉ dữ liệu (Zero Data Leakage cho real_11 và real_12)
train_inputs_joined = ' '.join(df_train['input'].tolist()).lower()
assert 'real_11' not in train_inputs_joined and 'real_12' not in train_inputs_joined, 'LỖI DATA LEAKAGE: real_11 hoặc real_12 xuất hiện trong train.csv!'
assert len(df_train) >= 261, f'Kỳ vọng >= 261 mẫu train (183 gốc + 26..28 real_pairs x3), thực tế: {len(df_train)}'
assert len(df_val) >= 24, f'Kỳ vọng >= 24 dòng val, thực tế: {len(df_val)}'
assert len(df_test) in (16, 17), f'Kỳ vọng 16 hoặc 17 dòng test (gồm real_11.jpg), thực tế: {len(df_test)}'
assert os.path.exists(f'{DATASET_ROOT}/processed/test/input/real_11.jpg'), 'Thiếu file datasets/processed/test/input/real_11.jpg'
assert os.path.exists(f'{DATASET_ROOT}/processed/val/input/real_12.jpg'), 'Thiếu file datasets/processed/val/input/real_12.jpg'

print('\n[OK] Kiểm tra chống Data Leakage: real_11.jpg và real_12.jpg hoàn toàn độc lập khỏi tập Train!')
print('\nThống kê theo split trong manifest:')
print(df_manifest.groupby('split').agg(
    inputs=('input_path', 'count'),
    unique_targets=('target_path', 'nunique'),
))

HAS_VAL  = len(df_val) > 0
HAS_TEST = len(df_test) > 0
MODE = 'full' if (HAS_VAL and HAS_TEST) else 'smoke'


In [ ]:
#@title 5. Khởi tạo Datasets & DataLoaders (Tự động lọc biến dạng hình học & tách riêng ảnh Unpaired Blind Evaluation)
train_ds = DocumentRestorationDataset(
    TRAIN_CSV,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    patch_size=(IMAGE_SIZE, IMAGE_SIZE),
    split='train',
    is_train=True,
)
val_ds = DocumentRestorationDataset(
    VAL_CSV,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    patch_size=(IMAGE_SIZE, IMAGE_SIZE),
    split='val',
    is_train=False,
) if HAS_VAL else None
test_ds = DocumentRestorationDataset(
    TEST_CSV,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    patch_size=(IMAGE_SIZE, IMAGE_SIZE),
    split='test',
    is_train=False,
) if HAS_TEST else None

assert train_ds.missing_count == 0, f'Lỗi missing file trong train.csv: {train_ds.missing_count}'
assert val_ds is None or val_ds.missing_count == 0, f'Lỗi missing file trong val.csv: {val_ds.missing_count}'
assert test_ds is None or test_ds.missing_count == 0, f'Lỗi missing file trong test.csv: {test_ds.missing_count}'
assert len(train_ds) > 0, 'Không tìm thấy cặp ảnh train hợp lệ!'

print(f'[TRAIN] Tổng CSV: {train_ds.total_rows} | Đã lọc hình học/Anh06: {train_ds.filtered_count} | Missing: {train_ds.missing_count} -> Sử dụng: {len(train_ds)} cặp chuẩn')
if val_ds:
    print(f'[VAL]   Tổng CSV: {val_ds.total_rows} | Đã lọc hình học: {val_ds.filtered_count} | Unpaired Blind (real_12): {val_ds.unpaired_count} | Missing: {val_ds.missing_count} -> Sử dụng tính PSNR: {len(val_ds)} cặp chuẩn')
if test_ds:
    print(f'[TEST]  Tổng CSV: {test_ds.total_rows} | Unpaired Blind (real_11): {test_ds.unpaired_count} | Missing: {test_ds.missing_count} -> Sử dụng tính PSNR: {len(test_ds)} cặp chuẩn')

sample_x, sample_y = train_ds[0]
print(f'\nTensor shape: input={tuple(sample_x.shape)}, target={tuple(sample_y.shape)} '
      f'(Target mean={sample_y.mean():.3f}, max={sample_y.max():.2f} — nền giấy đã chuẩn hóa trắng tinh)')

kw = dict(num_workers=2, pin_memory=(DEVICE.type == 'cuda'))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False, **kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **kw) if val_ds else None
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **kw) if test_ds else None

VIS_DS = val_ds if (val_ds is not None and len(val_ds) > 0) else train_ds
VIS_IDX = list(range(min(3, len(VIS_DS))))


In [ ]:
#@title 6. Khởi tạo Model U-Net & Nạp Trọng Số Warm-Start (models/restoration_best.pth) + DocumentCleanLoss (kèm BleedThroughSuppressionLoss)
model = DocumentRestorationUNet(in_channels=1, out_channels=1).to(DEVICE)
loss_fn = DocumentCleanLoss().to(DEVICE)
opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

LAST, BEST = f'{RUN_DIR}/restoration_last.pth', f'{RUN_DIR}/restoration_best.pth'
PRETRAINED_BEST = f'{MODELS_DIR}/restoration_best.pth'
LOG_JSON, LOG_CSV = f'{RUN_DIR}/training_log.json', f'{RUN_DIR}/training_log.csv'
history, start_epoch, best_psnr = [], 1, -float('inf')

WARM_START_FROM_BEST = True  #@param {type:"boolean"}
RESUME = False               #@param {type:"boolean"}

if RESUME and os.path.exists(f'{RUN_DIR}/resume.pt'):
    try:
        ck = torch.load(f'{RUN_DIR}/resume.pt', map_location=DEVICE)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        history, start_epoch, best_psnr = ck['history'], ck['epoch'] + 1, ck['best_psnr']
        print('Đã khôi phục (resume) từ epoch', ck['epoch'])
    except Exception as e:
        print('Bỏ qua checkpoint resume không tương thích:', e)
elif WARM_START_FROM_BEST and os.path.exists(PRETRAINED_BEST):
    state_dict = torch.load(PRETRAINED_BEST, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict, strict=False)
    shutil.copyfile(PRETRAINED_BEST, BEST)
    print(f'[Warm-Start] Đã nạp thành công trọng số pretrained từ: {PRETRAINED_BEST}')

print('Tổng số tham số U-Net 3 tầng:', f'{sum(p.numel() for p in model.parameters()):,}')


In [ ]:
#@title 7. Hàm đánh giá & Hiển thị trực quan ảnh Grayscale chuẩn [0, 1]
@torch.no_grad()
def evaluate(loader):
    model.eval(); tl = tp = n = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        p = model(x)
        tl += loss_fn(p, y).item(); tp += calculate_psnr(p, y); n += 1
    return tl / max(n, 1), tp / max(n, 1)

def to_gray_img(t):
    return t.detach().cpu().squeeze().clamp(0, 1).numpy()

@torch.no_grad()
def show_samples(title):
    model.eval()
    fig, ax = plt.subplots(len(VIS_IDX), 3, figsize=(12, 4 * len(VIS_IDX)), squeeze=False)
    for r, i in enumerate(VIS_IDX):
        x, y = VIS_DS[i]
        p = model(x.unsqueeze(0).to(DEVICE))[0]
        in_psnr = calculate_psnr(x, y)
        out_psnr = calculate_psnr(p.cpu(), y)
        cols = [
            (y, '1. Ground Truth (Clean White)'),
            (x, f'2. Degraded Input ({in_psnr:.2f} dB)'),
            (p, f'3. U-Net Restored ({out_psnr:.2f} dB)'),
        ]
        for c, (im, name) in enumerate(cols):
            ax[r][c].imshow(to_gray_img(im), cmap='gray', vmin=0.0, vmax=1.0)
            ax[r][c].set_title(name)
            ax[r][c].axis('off')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_history():
    if not history:
        return
    h = pd.DataFrame(history)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(h.epoch, h.train_loss, label='Train Loss', marker='o', markersize=3)
    if 'val_loss' in h and h.val_loss.notna().any():
        ax[0].plot(h.epoch, h.val_loss, label='Val Loss', marker='s', markersize=3)
    ax[0].set_title('DocumentCleanLoss'); ax[0].set_xlabel('Epoch'); ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].plot(h.epoch, h.train_psnr, label='Train PSNR', marker='o', markersize=3)
    if 'val_psnr' in h and h.val_psnr.notna().any():
        ax[1].plot(h.epoch, h.val_psnr, label='Val PSNR', marker='s', markersize=3)
    ax[1].set_title('PSNR (dB)'); ax[1].set_xlabel('Epoch'); ax[1].legend(); ax[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
#@title 8. Huấn luyện Model trên GPU (Tự động lưu Best Checkpoint vào Drive)
from IPython.display import clear_output

# Số vòng lặp random crop mỗi epoch
REPEAT_PER_EPOCH = 2 #@param {type:"integer"}

for epoch in range(start_epoch, EPOCHS + 1):
    model.train(); tl = tp = n = 0; t0 = time.time()
    for _ in range(REPEAT_PER_EPOCH):
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
                p = model(x)
            loss = loss_fn(p.float(), y.float())
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tl += loss.item()
            tp += calculate_psnr(p.detach().float(), y.float())
            n += 1
    scheduler.step()

    rec = dict(
        epoch=epoch,
        train_loss=round(tl / max(n, 1), 6),
        train_psnr=round(tp / max(n, 1), 2),
        val_loss=None,
        val_psnr=None,
        lr=round(opt.param_groups[0]['lr'], 7),
        seconds=round(time.time() - t0, 1),
    )
    if val_loader:
        vl, vp = evaluate(val_loader)
        rec['val_loss'], rec['val_psnr'] = round(vl, 6), round(vp, 2)
    history.append(rec)

    torch.save(model.state_dict(), LAST)
    shutil.copyfile(LAST, f'{MODELS_DIR}/restoration_last.pth')

    score = rec['val_psnr'] if val_loader else rec['train_psnr']
    if score > best_psnr:
        best_psnr = score
        torch.save(model.state_dict(), BEST)
        shutil.copyfile(BEST, f'{MODELS_DIR}/restoration_best.pth')

    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), history=history,
                    epoch=epoch, best_psnr=best_psnr), f'{RUN_DIR}/resume.pt')
    pd.DataFrame(history).to_csv(LOG_CSV, index=False)
    log_payload = dict(
        device=str(DEVICE),
        mode=MODE,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        learning_rate=LR,
        epochs=epoch,
        train_samples=len(train_ds),
        val_samples=len(val_ds) if val_ds else 0,
        best_val_psnr=round(best_psnr, 2),
        log=history,
    )
    with open(LOG_JSON, 'w', encoding='utf-8') as f:
        json.dump(log_payload, f, indent=2)
    shutil.copyfile(LOG_JSON, f'{MODELS_DIR}/training_log.json')

    print(f"Epoch [{epoch}/{EPOCHS}] ({rec['seconds']}s) | Train Loss: {rec['train_loss']:.4f} | Train PSNR: {rec['train_psnr']:.2f} dB"
          + (f" | Val Loss: {rec['val_loss']:.4f} | Val PSNR: {rec['val_psnr']:.2f} dB (Best: {best_psnr:.2f} dB)" if val_loader else ''))
    if epoch % VIS_EVERY == 0 or epoch == EPOCHS:
        clear_output(wait=True)
        plot_history()
        show_samples(f'Epoch {epoch}/{EPOCHS} — Best PSNR: {best_psnr:.2f} dB')

print('Huấn luyện hoàn tất! Checkpoint tốt nhất đã lưu tại:', BEST)


In [ ]:
#@title 9. Đánh giá Tập Test/Val & Kiểm chứng Thực lực 100% Pure U-Net trên `real_11.jpg` (Test) & `real_12.jpg` (Val)
BLIND_OUT_DIR = f'{PROJECT_DIR}/outputs/restoration/real_blind_test'
get_ipython().system(f'python training/evaluate_restoration.py --model "{BEST}" --test-csv "{TEST_CSV}" --val-csv "{VAL_CSV}" --include-val --blind-out-dir "{BLIND_OUT_DIR}"')
get_ipython().system(f'python training/infer_restoration.py --model "{BEST}" --input "{DATASET_ROOT}/real" --output "{PROJECT_DIR}/outputs/restoration_real" --preview-dir "{DATASET_ROOT}/real/unet_results_preview" --blind-out-dir "{BLIND_OUT_DIR}"')

# 1. Hiển thị to, rõ ràng kết quả Kiểm chứng Thực lực 100% Pure U-Net trên real_11.jpg (Test) và real_12.jpg (Val)
blind_comp_files = [
    f'{BLIND_OUT_DIR}/real_11_blind_comparison.png',
    f'{BLIND_OUT_DIR}/real_12_blind_comparison.png',
]
blind_comp_files = [p for p in blind_comp_files if os.path.exists(p)]
if blind_comp_files:
    print('\n' + '=' * 80)
    print('KIỂM CHỨNG THỰC LỰC 100% PURE U-NET (KHÔNG TARGET GIẢ, KHÔNG HẬU XỬ LÝ): real_11.jpg & real_12.jpg')
    print('=' * 80)
    fig, axes = plt.subplots(len(blind_comp_files), 1, figsize=(20, 8.5 * len(blind_comp_files)), squeeze=False)
    for idx, fpath in enumerate(blind_comp_files):
        bgr = cv2.imread(fpath)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        axes[idx][0].imshow(rgb)
        axes[idx][0].set_title(f'[BLIND TEST 100% PURE U-NET] {os.path.basename(fpath)}', fontsize=14, fontweight='bold')
        axes[idx][0].axis('off')
    plt.tight_layout()
    plt.show()

# 2. Hiển thị mẫu kết quả 3 cột của các ảnh thực tế có Target trong tập Train
preview_files = sorted(glob.glob(f'{DATASET_ROOT}/real/unet_results_preview/*_unet_preview.png'))
train_preview_files = [p for p in preview_files if 'real_11' not in os.path.basename(p) and 'real_12' not in os.path.basename(p)][:6]
if train_preview_files:
    print(f'\n=== MẪU KẾT QUẢ U-NET TRÊN ẢNH THỰC TẾ CÓ TARGET (3 CỘT: INPUT | TARGET | U-NET OUTPUT) ===')
    fig, axes = plt.subplots(len(train_preview_files), 1, figsize=(18, 5.5 * len(train_preview_files)), squeeze=False)
    for idx, fpath in enumerate(train_preview_files):
        bgr = cv2.imread(fpath)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        axes[idx][0].imshow(rgb)
        axes[idx][0].set_title(os.path.basename(fpath), fontsize=12, fontweight='bold')
        axes[idx][0].axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
#@title 10. Tải trọng số đã train (restoration_best.pth) về máy tính cá nhân
from google.colab import files
for f in [BEST, LOG_JSON]:
    if os.path.exists(f):
        files.download(f)
print('Các file cũng đã được lưu an toàn trên Google Drive tại:', RUN_DIR, 'và', MODELS_DIR)